# 딥러닝기초 파일별 2 · 01_pytorch (Day01)

**한 줄 요약** · Day01 강의 165셀을 원본 순서대로 정리 · 텐서 표현 → 선택 → 모양 → shape 계약 → 예측 · 손실 → 기울기 → 업데이트 · 학습률

- **원본** · `7. 딥러닝/00_딥러닝_기초/실습/01_pytorch.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/06_딥러닝기초·PyTorch` 딥러닝기초-1 1 ~ 7장 (같은 개념을 연습문제 P01 ~ P15로 정리)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 텐서를 만들고 shape · dtype · device를 축의 의미와 함께 읽는다
2. 행 · 열 · 구간 · 조건으로 꺼내고 reshape · transpose · squeeze · cat · stack을 골라 쓴다
3. `[n, 1] - [n]` 함정을 찾고, `*`와 `@`의 결과 shape를 계산 전에 말한다
4. MSE와 기울기를 계산하고 `backward()` · 누적 · reset · `no_grad`의 책임을 구분한다
5. 학습 한 단계를 직접 짜서 반복하고, 학습률에 따라 달라지는 loss 경로를 판정한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **학습 코드의 shape 에러를 읽을 때** · 행렬 곱 계약 · broadcasting (4장)
- **loss가 이상하게 크거나 안 줄 때** · 1:1 짝 · 기울기 누적 · 학습률 (4 · 6 · 7장)
- **모델에 샘플 하나를 넣을 때** · `[1]` vs `[1:2]` · unsqueeze (2 · 3장)
- **NumPy 전처리 결과를 모델에 넣을 때** · from_numpy · dtype (1 · 3장)
- **처음 보는 학습 코드를 읽을 때** · 역할 찾기 (8장)

## 학습 지도

<img src="images/pdl2_map.png" alt="01_pytorch 학습 지도" width="837">

**예상 학습 시간** · 개념 2시간 + 실습 1시간 (GPU 없이 모든 셀이 몇 초)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 구성</td></tr>
<tr><td>1</td><td>데이터를 텐서로 표현하기</td><td>틀린 예측을 어떻게 재고, 데이터를 어떤 그릇에 어떻게 담는가?</td></tr>
<tr><td>2</td><td>원하는 부분 선택하기</td><td>관측 × 특성 표에서 원하는 부분을 꺼내고, 축이 사라지는지 남는지 읽으려면?</td></tr>
<tr><td>3</td><td>같은 값을 다른 모양으로</td><td>모양을 바꾸고 합칠 때 원소 수와 축의 의미를 어떻게 지키는가?</td></tr>
<tr><td>4</td><td>실행됐는데 틀린 계산</td><td>에러 없이 틀리는 계산을 어떻게 찾고, 행렬 곱 결과를 어떻게 미리 말하는가?</td></tr>
<tr><td>5</td><td>예측과 틀린 정도</td><td>가장 작은 모델이 예측을 만들고, 그 예측이 얼마나 틀렸는지 숫자 하나로 요약하려면?</td></tr>
<tr><td>6</td><td>어느 방향으로 바꿀까</td><td>loss를 줄이는 방향을 어떻게 계산하고, backward는 무엇을 하고 무엇을 하지 않는가?</td></tr>
<tr><td>7</td><td>실제로 바꾸고 반복하기</td><td>parameter가 실제로 바뀌는 순간은 언제이고, 학습률은 경로를 어떻게 바꾸는가?</td></tr>
<tr><td>8</td><td>종합 실습</td><td>처음 보는 숫자와 처음 보는 변수 이름에서도 같은 학습 구조를 찾고 실행할 수 있는가?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

원본은 CPU만 씀 · 이 노트북도 그대로. **셀은 위에서부터 차례로 실행** (앞 섹션의 `weather` · `x_train` · `target_train`을 이어서 씀).

*항목 → 설정*

- **스레드** · `torch.set_num_threads(1)` · 작은 텐서라 1개로 충분 (이 PC에서 CPU가 바쁠 때 여러 스레드가 수십 배 느려진 적이 있음)
- **라이브러리** · `torch` (3-5에서 `numpy`)
- **seed** · 난수를 쓰는 1-2에서 `torch.manual_seed(7)` (원본과 같음)
- **장치** · CPU (원본과 같음 · GPU가 있어도 결과 같음)
- **원본 셀 수** · 165개 (학생 빈칸 셀 11개 · 그림 셀 13개 포함)

**실행 흐름**

1. `torch` 불러오기 · 버전 출력
2. 스레드 수 1개로 설정
3. 실행 위치 확인

In [1]:
# 실행 환경 확인: 원본 셀 6과 같은 import · 버전 · 실행 위치
# 왜 · 원본과 같은 환경인지 먼저 확인해 결과 차이의 원인을 줄이기 위해
import torch  # 텐서 · 자동 미분

print("PyTorch 버전:", torch.__version__.split("+")[0])  # 설치된 버전 기록
torch.set_num_threads(1)  # 작은 텐서는 스레드 1개로 충분 (다른 프로그램과 CPU 경쟁 방지)
print("오늘의 기본 실행 위치:", torch.device("cpu"))     # 원본은 CPU만으로 끝까지 실행

PyTorch 버전: 2.11.0
오늘의 기본 실행 위치: cpu


### 원본 구성

- **Prelude** · 컴퓨터는 틀린 예측에서 어떻게 나아질까? (예측 6 · 실제 10)
- **§1 ~ 3** · Tensor 표현 · 선택 · 모양 바꾸기 (짧은 실습 · 직접 확인하기 4개)
- **§4 ~ 5** · 실행됐는데 틀린 계산 · 행렬 곱 shape 계약 (낯선 코드에서 두 문제 찾기)
- **§6 ~ 7** · 가장 작은 모델 `ŷ = wx + b` · MSE
- **§8 ~ 9** · 국소 기울기 · Autograd · 수동 update · 학습 loop (새 숫자 적용 · 오류 찾기)
- **§10** · 학습률 · 전체 흐름 코드 읽기 · 틀린 문장 고치기 · Closing 자기 말 설명

**원본 힌트**

- 원본의 학생 빈칸 셀(`None` · `""` · `False`)은 모두 답을 채워 실행함
- 원본 그림(assets/*.svg)은 원본 폴더에 없어 새로 그림

---
## 1. 데이터를 텐서로 표현하기

> **이 장의 흐름**
>
> - 원본 Prelude의 '예측 6 · 실제 10'에서 시작
> - 값을 담는 그릇 Tensor의 shape · 축 의미 · dtype을 읽는 법으로

<img src="images/dl1_dims.png" alt="0 ~ 4차원 텐서의 shape" width="837">

**파이프라인 위치** · ① 데이터 준비 — 값을 텐서로 담고 모양 · 종류 확인

### 1-1. `A` 틀린 예측 재기 · `prediction - actual` <sub>원본 Prelude · 셀 2 ~ 4</sub>

> 🎯 **현재 규칙이 6을 예측했고 실제값이 10이면, 무엇을 먼저 숫자로 만들어야 하는가?**

💡 **쉽게 말하면**

- 학습은 **틀린 정도를 숫자로 재는 것**에서 시작.
- 차이의 부호는 예측이 큰지 작은지, 크기는 얼마나 틀렸는지를 알려 줌.

**실행 흐름**

1. 입력 · 현재 규칙 · 실제값
2. 예측 만들기
3. 차이 계산
4. w를 바꿔 다시 예측

**먼저 예측** · `3 × 2`의 예측과 실제값 10의 차이는? — 답은 코드 실행 결과 아래에

In [2]:
# 원본 Prelude(셀 3): 입력 2 · 현재 규칙 3 · 실제값 10
# 왜 · 학습이 '예측과 실제값의 차이를 줄이는 일'이라는 것을 숫자 하나로 먼저 보기 위해
input_value = 2.0     # 입력
current_weight = 3.0  # 지금의 내부 값 (규칙)
actual_value = 10.0   # 실제값 (정답)

# 왜 · 예측과 실제값의 차이를 직접 계산해 손실의 출발점을 보기 위해
prediction = current_weight * input_value  # 1. 예측 만들기
difference = prediction - actual_value     # 2. 실제값과 얼마나 다른지

print("입력:", input_value)      # 2.0
print("현재 예측:", prediction)  # 3 × 2
print("실제값:", actual_value)   # 10.0
print("차이:", difference)       # 음수 = 예측이 작음


# 3 · 4단계 미리보기: w를 조금 키우면 차이가 줄어드나?
# 왜 · 규칙(w)을 조금 바꾸면 차이가 줄어드는지 미리 확인해 경사하강의 감을 잡기 위해
for w_try in [3.0, 4.0, 5.0]:  # 내부 값 후보
    # 새 예측
    print(f"w={w_try} → 예측 {w_try * input_value} · 차이 {w_try * input_value - actual_value}")

입력: 2.0
현재 예측: 6.0
실제값: 10.0
차이: -4.0
w=3.0 → 예측 6.0 · 차이 -4.0
w=4.0 → 예측 8.0 · 차이 -2.0
w=5.0 → 예측 10.0 · 차이 0.0


**결과** · 예측 정답은 `-4.0` — 예측이 4 작음

- `현재 예측: 6.0` — 현재 규칙 3 × 입력 2
- `차이: -4.0` — 음수 → 예측이 실제값보다 작음
- `w=4.0 → 예측 8.0 · 차이 -2.0` — w를 키우자 차이가 줄어듦
- `w=5.0 → 예측 10.0 · 차이 0.0` — 5 × 2 = 10 · 이 데이터에 딱 맞는 규칙

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>차이 <code>prediction - actual</code></td><td>부호 = 방향 · 크기 = 얼마나</td><td>어느 쪽으로 고칠지 볼 때</td></tr>
<tr><td>후보를 하나씩 대 보기</td><td>w를 바꿔 다시 예측</td><td>값이 하나뿐인 예시 (이 섹션)</td></tr>
<tr><td>기울기로 고치기</td><td>지금 위치에서 방향을 계산</td><td>실제 모델 (6 · 7장)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 Prelude의 1 ~ 4단계 중 3단계(내부 값 바꾸기)는 아직 방법을 모름 → 6 · 7장에서 채움

> ✅ **한 줄 정리** · 학습의 출발은 예측과 실제값의 차이를 숫자로 만드는 것이다.

**딥러닝 코드에서는**

- 모든 학습 코드의 첫 두 줄 = `pred = model(x)` → `loss = criterion(pred, y)`

### 1-2. `A` 텐서 만들기 여섯 가지 · `torch.tensor()` · `zeros()` · `arange()` · `rand()` · `randn()` <sub>원본 §1 · 셀 6 ~ 9</sub>

> 🎯 **값을 알 때 · 구조만 필요할 때 · 임의의 시작값이 필요할 때 텐서를 어떻게 만드는가?**

💡 **쉽게 말하면**

- 함수 이름보다 **무엇을 담을지**를 먼저 정함.
- 값을 알면 직접, 틀만 필요하면 0 · 1, 무작위가 필요하면 난수 + seed.

**실행 흐름**

1. seed 7 고정
2. 여섯 함수로 만들기
3. shape · dtype 비교
4. arange · rand 값 확인

**먼저 예측** · `torch.arange(0, 6, 2)`의 값과 dtype은? — 답은 코드 실행 결과 아래에

In [3]:
# 원본 셀 8: 같은 seed로 여섯 가지 만들기
# 왜 · seed를 고정한 뒤 여러 방식으로 텐서를 만들어 결과를 다시 얻을 수 있게 하기 위해
torch.manual_seed(7)  # 난수를 다시 확인할 수 있게 고정

# 왜 · 값을 직접 지정한 텐서와 자동 생성 텐서를 비교하기 위해
known_values = torch.tensor([2.0, 4.0, 6.0])  # 값을 이미 알 때
zero_frame = torch.zeros(2, 3)                # 구조만 · 0으로
one_frame = torch.ones(2, 3)                  # 구조만 · 1로
ordered_values = torch.arange(0, 6, 2)        # 0부터 2씩 (6 제외)
uniform_samples = torch.rand(2, 3)            # 0 이상 1 미만 균등분포
normal_samples = torch.randn(2, 3)            # 평균 0 · 표준편차 1


# 값은 외우지 않고 shape · dtype만 비교
# 왜 · 난수 값보다 모양 · 자료형이 코드 오류를 가르는 기준이기 때문에
created = {"직접 지정": known_values, "0으로 준비": zero_frame,           # 이름 → 텐서
           "1로 준비": one_frame, "일정 간격": ordered_values,
           "균등 난수": uniform_samples, "정규 난수": normal_samples}
for label, value in created.items():                                      # 여섯 텐서를 차례로
    print(f"{label:8s} shape={tuple(value.shape)} dtype={value.dtype}")   # 모양 · 종류
print("arange 값:", ordered_values.tolist())                              # stop 6은 빠짐
print("rand 첫 행:", [round(v, 4) for v in uniform_samples[0].tolist()])  # seed 7 → 같은 환경에서 매번 같은 값

직접 지정    shape=(3,) dtype=torch.float32
0으로 준비   shape=(2, 3) dtype=torch.float32
1로 준비    shape=(2, 3) dtype=torch.float32
일정 간격    shape=(3,) dtype=torch.int64
균등 난수    shape=(2, 3) dtype=torch.float32
정규 난수    shape=(2, 3) dtype=torch.float32
arange 값: [0, 2, 4]
rand 첫 행: [0.5349, 0.1988, 0.6592]


**결과** · 예측 정답은 `[0, 2, 4]` · `torch.int64` — stop 6은 빠지고, 정수 인자라 정수

- `직접 지정 shape=(3,) dtype=torch.float32` — 실수를 넣어 float32
- `일정 간격 shape=(3,) dtype=torch.int64` — 정수만 → int64
- `arange 값: [0, 2, 4]` — 0부터 2씩 · 6 제외
- `rand 첫 행: [0.5349, 0.1988, 0.6592]` — seed 7이면 같은 환경에서 다시 실행해도 같은 값 (0 이상 1 미만)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>torch.tensor(값)</code></td><td>이미 아는 값</td><td>데이터 · 정답을 직접 넣을 때</td></tr>
<tr><td><code>zeros</code> · <code>ones</code></td><td>0 · 1로 채운 틀</td><td>결과를 담을 자리</td></tr>
<tr><td><code>arange(a, b, step)</code></td><td>규칙 수열 · stop 제외</td><td>번호 목록</td></tr>
<tr><td><code>rand</code></td><td>0 이상 1 미만 균등</td><td>0 ~ 1 무작위 값</td></tr>
<tr><td><code>randn</code></td><td>평균 0 · 표준편차 1 정규</td><td>잡음 · 가중치 초깃값</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - seed는 고정한 그 순간부터 같은 환경에서 같은 순서를 만듦 → 난수를 쓰는 셀마다 다시 고정

> ✅ **한 줄 정리** · 값을 알면 `tensor`, 틀이면 `zeros` · `ones`, 수열은 `arange`, 무작위는 `rand` · `randn`으로 만든다.

**딥러닝 코드에서는**

- `torch.manual_seed(42)` · 가중치 초깃값 `randn` · 결과 자리 `zeros(len(dataset))`

### 1-3. `A` shape와 축의 의미 · `shape` · `ndim` <sub>원본 §1 · 셀 10 ~ 16 · 19</sub>

> 🎯 **shape `(2, 3)`이 같은 두 텐서는 같은 계산에 넣어도 되는가?**

💡 **쉽게 말하면**

- shape는 **서랍장의 칸 수**만 알려 줌.
- 각 축이 학생인지 샘플인지는 데이터 설명을 보고 사람이 읽어야 함.

<img src="images/dl1_dims.png" alt="0 ~ 4차원 텐서의 shape" width="837">

**실행 흐름**

1. 0 ~ 4차원 텐서의 shape · ndim
2. 같은 shape · 다른 의미 비교
3. mini_images 축을 문장으로

**먼저 예측** · `mini_images` `(2, 3, 2, 2)`를 문장으로 읽으면? — 답은 코드 실행 결과 아래에

In [4]:
# 원본 셀 11: 0차원 ~ 4차원
# 왜 · 축 개수가 늘어날 때 shape가 어떻게 바뀌는지 한 번에 비교하기 위해
scalar = torch.tensor(7.0)                                               # 숫자 하나
vector = torch.tensor([18.0, 21.0, 25.0])                                # 축 1개
matrix = torch.tensor([[18.0, 45.0, 0.0],                                # 관측 2 × 특성 3
                       [21.0, 55.0, 1.0]])
mini_images = torch.arange(24, dtype=torch.float32).reshape(2, 3, 2, 2)  # (N, C, H, W)

for name, value in {"scalar": scalar, "vector": vector,                 # 네 텐서를 차례로
                    "matrix": matrix, "mini_images": mini_images}.items():
    print(f"{name:12s} shape={tuple(value.shape)}, ndim={value.ndim}")  # 축 길이 · 축 개수


# 원본 셀 15: 같은 shape · 다른 의미
# 왜 · 같은 shape라도 축의 의미가 다를 수 있다는 것을 확인하기 위해
student_subject_scores = torch.tensor([[80.0, 90.0, 70.0], [75.0, 85.0, 95.0]])  # 학생 2 × 과목 3
sample_class_scores = torch.tensor([[2.1, 0.3, -0.4], [0.2, 1.8, 0.1]])  # 샘플 2 × class 점수 3
print("shape만 같은가?:", student_subject_scores.shape == sample_class_scores.shape)  # 숫자만 비교


# 원본 셀 19 실습 답: mini_images 축을 문장으로
# 왜 · 4차원 이미지 텐서의 축을 말로 풀어 읽는 연습을 하기 위해
n, c, h, w = mini_images.shape                                 # 축 네 개를 이름으로 받기
print(f"샘플 {n}개, 샘플마다 채널 {c}개, 높이 {h}, 너비 {w}")  # 실습 빈칸 답

scalar       shape=(), ndim=0
vector       shape=(3,), ndim=1
matrix       shape=(2, 3), ndim=2
mini_images  shape=(2, 3, 2, 2), ndim=4
shape만 같은가?: True
샘플 2개, 샘플마다 채널 3개, 높이 2, 너비 2


**결과** · 예측 정답은 샘플 2개, 샘플마다 채널 3개, 높이 2, 너비 2

- `scalar shape=(), ndim=0` — 숫자 하나도 텐서 · 축 0개
- `mini_images shape=(2, 3, 2, 2), ndim=4` — [sample, channel, height, width]
- `shape만 같은가?: True` — 학생 × 과목 vs 샘플 × class 점수 → 숫자만 같음
- `샘플 2개, 샘플마다 채널 3개, 높이 2, 너비 2` — 원본 셀 19 실습 빈칸의 답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>.shape</code></td><td>축마다 길이</td><td>모양 확인 (가장 자주)</td></tr>
<tr><td><code>.ndim</code></td><td>축 개수</td><td>몇 차원인지</td></tr>
<tr><td>축 이름 (주석)</td><td>shape가 말해 주지 않는 의미</td><td>같은 계산에 넣어도 되는지 판단</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - shape가 같아도 서로 빼면 의미 없는 계산이 될 수 있음 → 데이터 설명과 연산이 기대하는 계약까지 확인

> ✅ **한 줄 정리** · shape는 축마다 길이만 말하므로, 각 축이 무엇인지 문장으로 함께 읽는다.

**딥러닝 코드에서는**

- `images.shape` → `(32, 3, 224, 224)` = 배치 · 채널 · 높이 · 너비

### 1-4. `A` 같은 숫자 · 다른 dtype · `dtype` · `device` <sub>원본 §1 · 셀 17 ~ 22</sub>

> 🎯 **화면에 똑같이 0 · 1 · 2가 보이면 두 텐서는 같은 역할인가?**

💡 **쉽게 말하면**

- dtype은 **숫자의 종류**, device는 **계산하는 작업장**.
- 연속 측정값은 실수, 범주 번호는 정수 · 함께 계산하려면 같은 작업장에 있어야 함.

**실행 흐름**

1. 정수 · 실수 텐서의 dtype · device
2. 0 · 1 · 2 두 가지
3. 무조건 float()의 결과

**먼저 예측** · `torch.tensor([0, 1, 2])`의 dtype은? — 답은 코드 실행 결과 아래에

In [5]:
# 원본 셀 17: 정수만 넣은 값 vs 실수
# 왜 · 정수만 넣으면 정수 텐서가 되어 계산 결과가 달라질 수 있음을 보기 위해
temperatures = torch.tensor([18, 21, 25])                                # 정수만
measurements = torch.tensor([[18.0, 45.0], [21.0, 55.0], [25.0, 60.0]])  # 실수
# 종류 · 위치 · 모양
print("temperatures:", temperatures.dtype, temperatures.device, tuple(temperatures.shape))
# 같은 세 가지
print("measurements:", measurements.dtype, measurements.device, tuple(measurements.shape))


# 원본 셀 21: 같은 0 · 1 · 2라도 역할이 다름
# 왜 · 같은 숫자라도 측정값 · 번호 · 정답처럼 역할에 따라 자료형이 달라야 함을 보이기 위해
measurement_values = torch.tensor([0.0, 1.0, 2.0])                    # 연속 측정값 → 실수
category_ids = torch.tensor([0, 1, 2])                                # 범주 번호 → 정수
for name, value in {"measurement_values": measurement_values,         # 두 텐서를 차례로
                    "category_ids": category_ids}.items():
    print(f"{name:18s} values={value.tolist()} dtype={value.dtype}")  # 값 · 종류


# 원본 셀 22 질문 3: 무조건 .float()를 붙이면 숨겨지는 것
# 왜 · 무조건 실수로 바꾸면 정답 번호의 의미가 숨겨지는 문제를 확인하기 위해
print("float 뒤:", category_ids.float().dtype, category_ids.float().tolist())  # 번호가 실수가 됨

temperatures: torch.int64 cpu (3,)
measurements: torch.float32 cpu (3, 2)
measurement_values values=[0.0, 1.0, 2.0] dtype=torch.float32
category_ids       values=[0, 1, 2] dtype=torch.int64
float 뒤: torch.float32 [0.0, 1.0, 2.0]


**결과** · 예측 정답은 `torch.int64` — 정수만 넣음

- `temperatures: torch.int64 cpu (3,)` — 정수만 넣어 int64
- `measurements: torch.float32 cpu (3, 2)` — 실수 → float32
- `category_ids values=[0, 1, 2] dtype=torch.int64` — 범주 번호 = 정수 인덱스 역할
- `float 뒤: torch.float32 [0.0, 1.0, 2.0]` — 번호가 실수가 되면 '범주'라는 역할이 숨겨짐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>int64</code></td><td>정수 · 범주 번호</td><td>분류 정답 (class 번호)</td></tr>
<tr><td><code>float32</code></td><td>실수 · 기울기 계산 가능</td><td>입력 · parameter · 예측 · MSE</td></tr>
<tr><td><code>.float()</code></td><td>실수로 바꾼 새 텐서</td><td>loss가 실수를 요구할 때만</td></tr>
<tr><td><code>.device</code></td><td>계산하는 곳 (cpu · cuda)</td><td>함께 계산할 값이 같은 곳인지</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 모든 텐서에 습관적으로 `.float()` → 범주 번호라는 정보가 사라짐

> ✅ **한 줄 정리** · dtype은 값의 역할에 맞추고, parameter · 예측 · MSE는 실수 텐서로 다룬다.

**딥러닝 코드에서는**

- `CrossEntropyLoss`의 정답은 int64 · `MSELoss`의 정답은 float32 (Day02)

---
## 2. 원하는 부분 선택하기

> **이 장의 흐름**
>
> - 1장에서 담은 표에서 이제 값을 꺼냄
> - 꺼낸 결과의 shape가 다음 연산의 계약과 맞는지까지 봄

<img src="images/pdl2_select.png" alt="weather 표에서 행 · 열 · 구간 · 조건" width="918">

**파이프라인 위치** · ③ 전처리 — 특성 · 샘플 고르기

### 2-1. `A` 행 · 열 · 구간 · 조건 · `weather[1]` · `weather[:, 0]` · `weather[mask]` <sub>원본 §2 · 셀 23 ~ 26 · 31 ~ 33</sub>

> 🎯 **관측 4 × 특성 3 표에서 한 관측 · 한 특성 · 구간 · 조건에 맞는 관측을 어떻게 꺼내는가?**

💡 **쉽게 말하면**

- `[행, 열]` 두 칸 주소.
- `:`는 '전부', 조건식은 True인 행만 남기는 체.

<img src="images/pdl2_select.png" alt="weather 표에서 행 · 열 · 구간 · 조건" width="918">

**실행 흐름**

1. 관측 4 × 특성 3 표
2. 행 · 열 · 구간 · 조건 꺼내기
3. 실습: 습도만

**먼저 예측** · 모든 관측의 습도만 꺼낸 결과의 shape는? — 답은 코드 실행 결과 아래에

In [6]:
# 원본 셀 24: 관측 4 × 특성 3 (온도 · 습도 · 비 여부)
# 왜 · 관측 × 특성 표에서 행 · 열의 의미를 정해 두기 위해
weather = torch.tensor([[18.0, 45.0, 0.0],   # 관측 1
                        [21.0, 55.0, 1.0],   # 관측 2
                        [25.0, 60.0, 1.0],   # 관측 3
                        [20.0, 50.0, 0.0]])  # 관측 4
feature_names = ["온도", "습도", "비 여부"]  # axis 1의 이름
print("shape:", tuple(weather.shape))        # (4, 3)


# 원본 셀 25: 행 · 열 · 구간 · 조건
# 왜 · 행 · 열 · 구간 · 조건으로 필요한 값을 꺼내는 방법을 비교하기 위해
print("두 번째 관측 전체:", weather[1].tolist())               # axis 0에서 하나
print("모든 관측의 온도:", weather[:, 0].tolist())             # 모든 행 · 열 0
print("앞의 두 관측, 온도와 습도:", weather[:2, :2].tolist())  # 두 축 범위
print("비가 온 관측:", weather[weather[:, 2] == 1].tolist())   # 조건 True인 행만


# 원본 셀 31 ~ 33 실습: 모든 관측의 습도만
# 왜 · 특성 하나만 모든 관측에서 꺼내는 대표 상황을 연습하기 위해
humidity_answer = weather[:, feature_names.index("습도")]                         # 열 위치 1
print("습도:", humidity_answer.tolist(), "shape:", tuple(humidity_answer.shape))  # (4,)

shape: (4, 3)
두 번째 관측 전체: [21.0, 55.0, 1.0]
모든 관측의 온도: [18.0, 21.0, 25.0, 20.0]
앞의 두 관측, 온도와 습도: [[18.0, 45.0], [21.0, 55.0]]
비가 온 관측: [[21.0, 55.0, 1.0], [25.0, 60.0, 1.0]]
습도: [45.0, 55.0, 60.0, 50.0] shape: (4,)


**결과** · 예측 정답은 `(4,)` — 열 하나를 정수로 골라 특성 축이 사라짐

- `두 번째 관측 전체: [21.0, 55.0, 1.0]` — axis 0에서 위치 1
- `모든 관측의 온도: [18.0, 21.0, 25.0, 20.0]` — 모든 행 · 열 0
- `비가 온 관측: [[21.0, 55.0, 1.0], [25.0, 60.0, 1.0]]` — 세 번째 특성이 1인 행만
- `습도: [45.0, 55.0, 60.0, 50.0] shape: (4,)` — 원본 셀 32 실습의 답 · 열 위치 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>weather[1]</code></td><td>한 관측 (행)</td><td>샘플 하나</td></tr>
<tr><td><code>weather[:, 0]</code></td><td>모든 관측의 한 특성 (열)</td><td>특성 하나만</td></tr>
<tr><td><code>weather[:2, :2]</code></td><td>두 축 모두 범위</td><td>일부 직사각형</td></tr>
<tr><td><code>weather[weather[:, 2] == 1]</code></td><td>조건 True인 행</td><td>조건에 맞는 샘플만</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 열 이름을 숫자로 외우면 틀리기 쉬움 → `feature_names.index("습도")`처럼 이름에서 위치를 찾기

> ✅ **한 줄 정리** · `[행, 열]` 주소에 위치 · 범위 · 조건을 넣어 원하는 부분을 꺼낸다.

**딥러닝 코드에서는**

- `X[:, 0]` (첫 특성) · `X[y == 1]` (정답 1인 샘플)

### 2-2. `A` 축이 사라지나 남나 · `weather[1]` · `weather[1:2]` <sub>원본 §2 · 셀 27 ~ 30</sub>

> 🎯 **`weather[1]`과 `weather[1:2]`는 같은 값인데 무엇이 다른가?**

💡 **쉽게 말하면**

- 정수 하나로 고르면 **그 축이 사라지고**, 범위로 고르면 **길이 1로 남음**.
- 다음 연산이 `[feature]`를 원하는지 `[batch, feature]`를 원하는지로 고름.

**실행 흐름**

1. 정수로 고르기
2. 범위로 고르기
3. 값 · shape 비교
4. unsqueeze(0)와 비교

**먼저 예측** · 두 결과의 shape는? — 답은 코드 실행 결과 아래에

In [7]:
# 원본 셀 28: 같은 값 · 다른 shape
# 왜 · 같은 값이라도 축 수가 다르면 모델이 다르게 받아들인다는 것을 보이기 위해
one_observation = weather[1]                                      # 정수 하나 → 관측 축 사라짐
one_observation_batch = weather[1:2]                              # 범위 → 길이 1 관측 축 유지
print("weather[1] 값:", one_observation.tolist())                 # 세 값
print("weather[1] shape:", tuple(one_observation.shape))          # (3,)
print("weather[1:2] 값:", one_observation_batch.tolist())         # 같은 세 값을 한 겹 더
print("weather[1:2] shape:", tuple(one_observation_batch.shape))  # (1, 3)


# 원본 셀 30 빈칸 답: [feature]를 기대하면 [1], [batch, feature]면 [1:2]
# 왜 · 모델이 기대하는 모양에 따라 인덱싱 방법을 골라야 하기 때문에
# 1 vs 2
print("특성 벡터:", one_observation.ndim, "차원 | 배치 1개:", one_observation_batch.ndim, "차원")
# 축 추가 = 범위 선택
print("unsqueeze(0)로도 같음:", torch.equal(one_observation.unsqueeze(0), one_observation_batch))

weather[1] 값: [21.0, 55.0, 1.0]
weather[1] shape: (3,)
weather[1:2] 값: [[21.0, 55.0, 1.0]]
weather[1:2] shape: (1, 3)
특성 벡터: 1 차원 | 배치 1개: 2 차원
unsqueeze(0)로도 같음: True


**결과** · 예측 정답은 `(3,)`과 `(1, 3)`

- `weather[1] shape: (3,)` — 관측 축 사라짐 → 특성 벡터
- `weather[1:2] shape: (1, 3)` — 관측 축 길이 1로 남음 → 배치 1개
- `unsqueeze(0)로도 같음: True` — 축을 나중에 붙여도 같은 결과

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>x[1]</code></td><td>축 사라짐 · <code>(3,)</code></td><td>특성 벡터 하나로 쓸 때</td></tr>
<tr><td><code>x[1:2]</code></td><td>축 유지 · <code>(1, 3)</code></td><td>배치 모양을 지켜야 할 때</td></tr>
<tr><td><code>x[1].unsqueeze(0)</code></td><td>사라진 축을 다시 붙임</td><td>한 샘플을 모델에 넣을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 둘 중 하나가 언제나 옳은 것은 아님 → 다음 연산이 기대하는 shape를 먼저 확인

> ✅ **한 줄 정리** · 정수로 고르면 축이 사라지고 범위로 고르면 길이 1로 남으므로, 다음 연산의 계약에 맞춰 고른다.

**딥러닝 코드에서는**

- `model(x[0:1])` 또는 `model(x[0].unsqueeze(0))` — 샘플 하나도 배치 모양으로

---
## 3. 같은 값을 다른 모양으로

> **이 장의 흐름**
>
> - 2장에서 꺼낸 값을 모델 입력 모양에 맞춤
> - 실행 가능 · 의미 보존 · 축 교환을 나눠 판정

<img src="images/dl1_reshape_t.png" alt="reshape(4, 3)과 .T는 같은 shape · 다른 값 배치" width="814">

**파이프라인 위치** · ③ 전처리 — 모델 입력 shape 만들기

### 3-1. `A` 원소 수 계약 · `reshape()` · `-1` · `numel()` <sub>원본 §3 · 셀 34 ~ 36</sub>

> 🎯 **모양을 바꿀 때 반드시 지켜야 하는 계약은 무엇이고, `-1`은 무엇을 대신 계산하는가?**

💡 **쉽게 말하면**

- reshape는 **같은 구슬을 다른 판에 다시 꽂기**.
- 칸 수(원소 수)가 같아야 하고, `-1`은 남은 칸 수를 계산해 주는 빈칸.

**실행 흐름**

1. 24개 → (2, 3, 4)
2. 샘플별 펼치기 (2, -1)
3. (3, -1)
4. 25칸에 넣으면 실패

**먼저 예측** · `blocks.reshape(3, -1)`의 shape는? — 답은 코드 실행 결과 아래에

In [8]:
# 원본 셀 35: 24개 값의 모양만 바꾸기
# 왜 · 값은 그대로 두고 모양만 바꾸는 여러 방법을 비교하기 위해
sequence = torch.arange(24)                   # 0 ~ 23
blocks = sequence.reshape(2, 3, 4)            # 2 × 3 × 4 = 24
flattened_per_sample = blocks.reshape(2, -1)  # 샘플은 두고 나머지 펴기
b = blocks.reshape(3, -1)                     # 24 ÷ 3 = 8

for name, value in [("원본", sequence), ("blocks", blocks),                  # 네 텐서를 차례로
                    ("샘플별 펼침", flattened_per_sample), ("reshape(3, -1)", b)]:
    print(f"{name:14s} shape={tuple(value.shape)} 원소 수={value.numel()}")  # 원소 수는 그대로


# 원소 수가 맞지 않으면 실패
# 왜 · reshape가 실패하는 조건을 에러로 확인하기 위해
try:
    sequence.reshape(5, 5)                                               # 25칸 ≠ 24개
except RuntimeError as error:                                            # 에러를 잡아 멈추지 않게
    print("reshape(5, 5) → RuntimeError:", str(error).split(" is ")[0])  # 메시지 앞부분

원본             shape=(24,) 원소 수=24
blocks         shape=(2, 3, 4) 원소 수=24
샘플별 펼침         shape=(2, 12) 원소 수=24
reshape(3, -1) shape=(3, 8) 원소 수=24
reshape(5, 5) → RuntimeError: shape '[5, 5]'


**결과** · 예측 정답은 `(3, 8)` — 24 ÷ 3 = 8

- `blocks shape=(2, 3, 4) 원소 수=24` — 2 × 3 × 4 = 24
- `샘플별 펼침 shape=(2, 12) 원소 수=24` — 샘플 축은 두고 나머지를 한 줄로
- `reshape(3, -1) shape=(3, 8) 원소 수=24` — 원본 셀 35의 b
- `reshape(5, 5) → RuntimeError: shape '[5, 5]'` — 25칸 ≠ 24개

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>reshape(2, 3, 4)</code></td><td>모든 축 직접</td><td>모양이 정해져 있을 때</td></tr>
<tr><td><code>reshape(2, -1)</code></td><td>한 축 자동</td><td>샘플 수만 정할 때</td></tr>
<tr><td><code>.numel()</code></td><td>원소 수</td><td>reshape 전후 비교</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `-1`은 한 자리에만 · 나누어떨어지지 않으면 실패

> ✅ **한 줄 정리** · reshape는 원소 수가 같을 때만 성공하며, `-1`은 남은 한 축의 길이를 계산한다.

**딥러닝 코드에서는**

- `x.reshape(x.shape[0], -1)` — 배치는 두고 펴기 (CNN 출력 → Linear 입력)

### 3-2. `A` reshape와 transpose · `reshape(4, 3)` · `.T` <sub>원본 §3 · 셀 36 ~ 38</sub>

> 🎯 **학생 3 × 문항 4를 `reshape(4, 3)`하면 문항 × 학생이 되는가?**

💡 **쉽게 말하면**

- reshape는 **순서대로 다시 채우기**, `.T`는 **가로 세로 돌려 보기**.
- 둘 다 `(4, 3)`이지만 값 배치가 다름.

<img src="images/dl1_reshape_t.png" alt="reshape(4, 3)과 .T는 같은 shape · 다른 값 배치" width="814">

**실행 흐름**

1. (3, 4) 표
2. reshape(4, 3) · .T 첫 행 비교
3. 같은지 확인
4. 세 기준 판정

**먼저 예측** · `reshape(4, 3)`과 `.T`의 첫 행은? — 답은 코드 실행 결과 아래에

In [9]:
# 원본 셀 37: 학생 3 × 문항 4
# 왜 · reshape가 축을 바꾸지 않고 값을 순서대로 다시 채운다는 것을 확인하기 위해
score_grid = torch.arange(1, 13).reshape(3, 4)         # 1 ~ 12
possible_but_reinterpreted = score_grid.reshape(4, 3)  # 순서대로 다시 채움
axes_swapped = score_grid.T                            # 행 ↔ 열 교환

# 왜 · reshape와 전치의 결과가 다르다는 것을 첫 행으로 비교하기 위해
print("원본 첫 행:", score_grid[0].tolist())         # 학생 1의 문항 4개
print("reshape(4, 3) 첫 행:", possible_but_reinterpreted[0].tolist())  # 앞에서부터 3개
print("transpose 첫 행:", axes_swapped[0].tolist())  # 원래 첫 열 = 문항 1의 학생 3명
# 같은 shape · 다른 값
print("두 결과의 값 배치가 같은가?:", torch.equal(possible_but_reinterpreted, axes_swapped))


# 원본 셀 38: 세 기준(실행 가능 · 의미 보존 · 축 교환)으로 판정
# 왜 · 실행 가능 여부와 의미 보존을 따로 판정해야 한다는 것을 확인하기 위해
for label, shape in [("reshape(3, 4)", (3, 4)), ("reshape(2, 6)", (2, 6))]:  # 두 후보
    ok = score_grid.numel() == shape[0] * shape[1]                           # 원소 수 계약
    print(f"{label}: 실행 가능 {ok}")                                        # 둘 다 12칸

원본 첫 행: [1, 2, 3, 4]
reshape(4, 3) 첫 행: [1, 2, 3]
transpose 첫 행: [1, 5, 9]
두 결과의 값 배치가 같은가?: False
reshape(3, 4): 실행 가능 True
reshape(2, 6): 실행 가능 True


**결과** · 예측 정답은 `[1, 2, 3]`과 `[1, 5, 9]` — 다름

- `reshape(4, 3) 첫 행: [1, 2, 3]` — 앞에서부터 3개씩 다시 채움
- `transpose 첫 행: [1, 5, 9]` — 원래 첫 열 = 문항 1의 학생 3명
- `두 결과의 값 배치가 같은가?: False` — 같은 shape · 다른 의미
- `reshape(2, 6): 실행 가능 True` — 실행은 되지만 '학생 × 문항' 의미는 깨짐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>reshape</code></td><td>원소 수만 맞으면 실행 · 의미는 보장 안 함</td><td>펴기 · 묶기</td></tr>
<tr><td><code>.T</code> · <code>transpose</code></td><td>축 순서를 바꿈</td><td>행 ↔ 열 의미를 바꿀 때</td></tr>
<tr><td>판정 세 기준</td><td>실행 가능 → 의미 보존 → 축 교환</td><td>모양을 바꾸기 전 점검</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `reshape(4, 3)`을 transpose라고 부르면 안 됨

> ✅ **한 줄 정리** · reshape는 순서대로 다시 채울 뿐이므로, 축의 의미를 바꾸려면 `.T` · `transpose`를 쓴다.

**딥러닝 코드에서는**

- 이미지 `(H, W, C)` → `(C, H, W)`는 `permute(2, 0, 1)` (reshape로 하면 그림이 뒤섞임)

### 3-3. `A` 길이 1 축 넣고 빼기 · `unsqueeze()` · `squeeze()` <sub>원본 §3 · 셀 39 · 40 · 46 · 47</sub>

> 🎯 **`(4,)` 정답을 `(4, 1)`로, 12개 값을 샘플 3 × 특성 4로 바꾸려면?**

💡 **쉽게 말하면**

- 길이 1 축은 **빈 포장 상자**.
- `unsqueeze`는 한 겹 싸고, `squeeze`는 벗김.

**실행 흐름**

1. (4,) → unsqueeze(1)
2. → squeeze(1)
3. 실습: 12개 → (3, 4)

**먼저 예측** · `target_flat.unsqueeze(1)`의 shape는? — 답은 코드 실행 결과 아래에

In [10]:
# 원본 셀 39: 길이 1 축 넣고 빼기
# 왜 · 길이 1 축을 넣고 빼서 모델이 원하는 모양에 맞추기 위해
target_flat = torch.tensor([3.0, 5.0, 7.0, 9.0])    # 정답 (4,)
target_column = target_flat.unsqueeze(1)            # 끝에 길이 1 축
target_restored = target_column.squeeze(1)          # 그 축을 다시 제거
print("flat:", tuple(target_flat.shape))            # (4,)
print("unsqueeze(1):", tuple(target_column.shape))  # (4, 1)
print("squeeze(1):", tuple(target_restored.shape))  # (4,)


# 원본 셀 46 ~ 47 실습 답: 12개 → 샘플 3 × 특성 4
# 왜 · 값 12개를 샘플 × 특성 모양으로 바꾸는 연습을 하기 위해
practice_values = torch.arange(12)                # 0 ~ 11
practice_matrix = practice_values.reshape(3, -1)  # 둘째 축은 자동 4
print("shape:", tuple(practice_matrix.shape), "원소 수:", practice_matrix.numel())  # 12 유지

flat: (4,)
unsqueeze(1): (4, 1)
squeeze(1): (4,)
shape: (3, 4) 원소 수: 12


**결과** · 예측 정답은 `(4, 1)`

- `unsqueeze(1): (4, 1)` — 끝에 길이 1 축
- `squeeze(1): (4,)` — 그 축을 다시 제거
- `shape: (3, 4) 원소 수: 12` — 원본 셀 47 실습 답 · `reshape(3, -1)`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>unsqueeze(dim)</code></td><td>길이 1 축 추가</td><td>정답을 예측 <code>(N, 1)</code>에 맞출 때</td></tr>
<tr><td><code>squeeze(dim)</code></td><td>길이 1 축 제거</td><td>출력 <code>(N, 1)</code> → <code>(N,)</code></td></tr>
<tr><td><code>reshape(3, -1)</code></td><td>원소 수 안에서 새 모양</td><td>샘플 × 특성으로 바꿀 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `squeeze()`를 dim 없이 쓰면 길이 1인 축이 모두 사라짐 → 항상 dim 지정

> ✅ **한 줄 정리** · 길이 1 축은 `unsqueeze(dim)`로 넣고 `squeeze(dim)`로 빼며, 축의 의미를 확인한다.

**딥러닝 코드에서는**

- `model(img.unsqueeze(0))` · `loss(pred.squeeze(1), y)`

### 3-4. `A` 기존 축 잇기와 새 축 쌓기 · `torch.cat()` · `torch.stack()` <sub>원본 §3 · 셀 41 ~ 45</sub>

> 🎯 **두 센서 기록을 하나의 긴 시간 흐름으로 이을지, 센서 축을 새로 만들지 어떻게 정하는가?**

💡 **쉽게 말하면**

- `cat`은 **기차 칸 이어 붙이기**(시간이 길어짐).
- `stack`은 **같은 기차를 나란히 세우기**(센서 번호 축이 생김).

**실행 흐름**

1. 오전 · 오후 cat · stack
2. 센서 A · B 이어 붙이기 · 쌓기
3. 새 축 번호로 센서 B 꺼내기

**먼저 예측** · `stack([sensor_a, sensor_b], dim=0)`의 shape는? — 답은 코드 실행 결과 아래에

In [11]:
# 원본 셀 41: 오전 · 오후 기록 (시간 2 × 특성 2)
# 왜 · 같은 모양의 텐서를 새 축으로 쌓을지 기존 축으로 이을지 비교하기 위해
morning = torch.tensor([[18.0, 45.0], [21.0, 55.0]])                    # 오전 두 관측
afternoon = torch.tensor([[25.0, 60.0], [20.0, 50.0]])                  # 오후 두 관측
print("cat:", tuple(torch.cat([morning, afternoon], dim=0).shape))      # 기존 축 늘리기
print("stack:", tuple(torch.stack([morning, afternoon], dim=0).shape))  # 새 축 만들기


# 원본 셀 44: 두 센서 기록
# 왜 · 두 센서 기록을 합칠 때 축 선택에 따라 의미가 달라지는 것을 보기 위해
sensor_a = torch.tensor([[10.0, 0.1], [11.0, 0.2]])                  # 센서 A
sensor_b = torch.tensor([[20.0, 0.3], [21.0, 0.4]])                  # 센서 B
long_timeline = torch.cat([sensor_a, sensor_b], dim=0)               # 하나의 긴 시간 흐름
separate_sensors = torch.stack([sensor_a, sensor_b], dim=0)          # 센서 축을 새로
# 시간 4 × 특성 2
print("기존 시간 축을 늘린 결과:", tuple(long_timeline.shape), "ndim", long_timeline.ndim)
# 센서 2 × 시간 2 × 특성 2
print("새 센서 축을 만든 결과:", tuple(separate_sensors.shape), "ndim", separate_sensors.ndim)
print("센서 B만:", separate_sensors[1, :, 0].tolist(), "(첫 특성)")  # 새 축 번호 1 = 센서 B

cat: (4, 2)
stack: (2, 2, 2)
기존 시간 축을 늘린 결과: (4, 2) ndim 2
새 센서 축을 만든 결과: (2, 2, 2) ndim 3
센서 B만: [20.0, 21.0] (첫 특성)


**결과** · 예측 정답은 `(2, 2, 2)` — 센서 × 시간 × 특성

- `cat: (4, 2)` — 시간 2 + 2
- `stack: (2, 2, 2)` — 맨 앞에 새 축
- `기존 시간 축을 늘린 결과: (4, 2) ndim 2` — 시간 × 특성
- `새 센서 축을 만든 결과: (2, 2, 2) ndim 3` — 센서 × 시간 × 특성
- `센서 B만: [20.0, 21.0] (첫 특성)` — 새 축 번호 1 = 센서 B

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>torch.cat(목록, dim)</code></td><td>기존 축을 늘림</td><td>시간 · 샘플을 이어 붙일 때</td></tr>
<tr><td><code>torch.stack(목록, dim)</code></td><td>새 축을 만듦 · 모양이 같아야</td><td>묶음을 구분할 때 (배치 만들기)</td></tr>
<tr><td>판정 질문</td><td>새 축의 이름을 말할 수 있나?</td><td>말할 수 없으면 stack이 아님</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 둘 다 실행된다고 같은 의미가 아님 → 결과 축 이름을 문장으로 말해 보기

> ✅ **한 줄 정리** · 기존 축을 늘리면 `cat`, 묶음을 구분할 새 축이 필요하면 `stack`을 쓴다.

**딥러닝 코드에서는**

- `DataLoader`가 샘플을 `stack`해 배치 축을 만듦 · 예측 모으기는 `torch.cat`

### 3-5. `B` NumPy와 오가기 · `torch.from_numpy()` · `.numpy()` <sub>원본 §3 · 셀 48 · 49</sub>

> 🎯 **이미 있는 NumPy 배열을 텐서로 가져오면 원본과 어떤 관계가 되는가?**

💡 **쉽게 말하면**

- `from_numpy`는 **같은 종이를 함께 보기**, `torch.tensor`는 **복사본 만들기**.

**실행 흐름**

1. float32 배열
2. from_numpy · numpy()
3. 원본 수정 → 공유 확인

**먼저 예측** · 원본 배열을 고치면 `from_numpy` 텐서도 바뀔까? — 답은 코드 실행 결과 아래에

In [12]:
# 원본 셀 49: NumPy ↔ 텐서
# 왜 · NumPy 배열과 텐서를 오가는 방법을 확인하기 위해
import numpy as np  # NumPy 배열

# 왜 · 자료형이 그대로 이어지는지 확인하기 위해
numpy_values = np.array([1.5, 2.5, 3.5], dtype=np.float32)  # 모델용 float32로 만든 배열
torch_values = torch.from_numpy(numpy_values)               # 배열 → 텐서 (메모리 공유)
back_to_numpy = torch_values.numpy()                        # 텐서 → 배열
print(type(numpy_values).__name__, numpy_values.dtype)      # ndarray float32
print(type(torch_values).__name__, torch_values.dtype)      # Tensor torch.float32
print(type(back_to_numpy).__name__, back_to_numpy.dtype)    # 다시 ndarray


# 보강: from_numpy는 메모리를 공유 · torch.tensor는 복사
# 왜 · 한쪽을 고치면 다른 쪽도 바뀌는 공유 함정을 미리 보기 위해
copied = torch.tensor(numpy_values)          # 수정 전에 복사해 둔 텐서
numpy_values[0] = 100.0                      # 원본 배열만 수정
print("from_numpy:", torch_values.tolist())  # 공유 → 100이 보임
print("tensor 복사:", copied.tolist())       # 복사본은 그대로

ndarray float32
Tensor torch.float32
ndarray float32
from_numpy: [100.0, 2.5, 3.5]
tensor 복사: [1.5, 2.5, 3.5]


**결과** · 예측 정답은 바뀜 — 메모리를 함께 씀

- `Tensor torch.float32` — float32 배열 → float32 텐서
- `from_numpy: [100.0, 2.5, 3.5]` — 원본 수정이 그대로 보임 (공유)
- `tensor 복사: [1.5, 2.5, 3.5]` — 수정 전에 복사한 텐서는 그대로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>torch.from_numpy(a)</code></td><td>메모리 공유 · 복사 없음</td><td>큰 배열을 넘길 때</td></tr>
<tr><td><code>torch.tensor(a)</code></td><td>복사</td><td>원본을 따로 지킬 때</td></tr>
<tr><td><code>t.numpy()</code></td><td>CPU 텐서 → 배열 (공유)</td><td>그래프 · NumPy 계산</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - NumPy 기본 실수는 float64 → 모델에 넣기 전 float32로 (원본은 처음부터 float32로 만듦)

> ✅ **한 줄 정리** · `from_numpy`는 원본과 메모리를 공유하고 `torch.tensor`는 복사한다.

**딥러닝 코드에서는**

- `torch.from_numpy(img).float()` · 결과 확인 `pred.detach().cpu().numpy()`

---
## 4. 실행됐는데 틀린 계산

> **이 장의 흐름**
>
> - 3장에서 모양을 맞춘 값으로 계산
> - 1:1 짝과 행렬 곱의 안쪽 · 바깥 축을 계산 전에 확인

<img src="images/pdl2_broadcast.png" alt="(4, 1) - (4,)는 16개 조합 · 1:1은 대각선 4개" width="832">

**파이프라인 위치** · ④ 모델 · ⑤ 학습 — 층 계산 · loss 직전 점검

### 4-1. `A` 실행됨 ≠ 의도한 계산 · `broadcasting` <sub>원본 §4 · 셀 50 ~ 57</sub>

> 🎯 **예측 `[4, 1]`에서 정답 `[4]`를 빼면 왜 에러 없이 `(4, 4)`가 되는가?**

💡 **쉽게 말하면**

- broadcasting은 **길이 1인 축을 늘려 맞추는 규칙**.
- 예측은 세로, 정답은 가로라 1:1 짝이 아니라 **모든 조합**이 됨.

<img src="images/pdl2_broadcast.png" alt="(4, 1) - (4,)는 16개 조합 · 1:1은 대각선 4개" width="832">

**실행 흐름**

1. (4, 1) - (4,)
2. 첫 행 확인
3. 정답 unsqueeze(1)
4. 의도한 broadcasting 세 경우

**먼저 예측** · `prediction_column - target_flat`의 shape는? — 답은 코드 실행 결과 아래에

In [13]:
# 원본 셀 51: 예측 [n, 1] · 정답 [n]
# 왜 · 예측 [n, 1]과 정답 [n]의 모양 차이가 조용한 버그를 만드는 상황을 재현하기 위해
prediction_column = torch.tensor([[2.0], [4.0], [6.0], [8.0]])  # (4, 1)
target_flat = torch.tensor([3.0, 5.0, 7.0, 9.0])                # (4,)
silent_difference = prediction_column - target_flat             # 에러 없이 계산됨
print("difference shape:", tuple(silent_difference.shape))      # (4, 4) 모든 조합
print("첫 행:", silent_difference[0].tolist())                  # 예측 2가 정답 4개와 비교됨


# 원본 셀 53: 정답에 축 추가 → 1:1
# 왜 · 정답에 축을 하나 더해 예측과 1:1로 맞추기 위해
target_column = target_flat.unsqueeze(1)             # (4, 1)
safe_difference = prediction_column - target_column  # 같은 위치끼리
# 모두 -1
print("1:1 difference shape:", tuple(safe_difference.shape), safe_difference.squeeze(1).tolist())


# 원본 셀 55 · 56 · 57: 의도한 broadcasting
# 왜 · 의도한 브로드캐스팅과 의도하지 않은 브로드캐스팅을 구분하기 위해
m1 = torch.tensor([[3.0, 3.0]])                                 # (1, 2)
m2 = torch.tensor([[2.0, 2.0]])                                 # (1, 2)
print("(1,2)+(1,2):", (m1 + m2).tolist())                       # 같은 모양
m3 = torch.tensor([[1.0, 2.0]]) + torch.tensor([[3.0], [4.0]])  # (1, 2) + (2, 1)
print("(1,2)+(2,1):", tuple(m3.shape), m3.tolist())             # 둘 다 늘어나 (2, 2)
# 모든 값에 0.5
print("행렬 + scalar:", (torch.tensor([[1.0, 2.0], [3.0, 4.0]]) + torch.tensor(0.5)).tolist())

difference shape: (4, 4)
첫 행: [-1.0, -3.0, -5.0, -7.0]
1:1 difference shape: (4, 1) [-1.0, -1.0, -1.0, -1.0]
(1,2)+(1,2): [[5.0, 5.0]]
(1,2)+(2,1): (2, 2) [[4.0, 5.0], [5.0, 6.0]]
행렬 + scalar: [[1.5, 2.5], [3.5, 4.5]]


**결과** · 예측 정답은 `(4, 4)` — 모든 조합 비교

- `difference shape: (4, 4)` — 16개 차이 · 에러 없음
- `첫 행: [-1.0, -3.0, -5.0, -7.0]` — 예측 2가 정답 4개 모두와 비교됨
- `1:1 difference shape: (4, 1) [-1.0, -1.0, -1.0, -1.0]` — 같은 위치끼리 → 모두 -1
- `(1,2)+(2,1): (2, 2) [[4.0, 5.0], [5.0, 6.0]]` — 둘 다 늘어나는 의도한 경우
- `행렬 + scalar: [[1.5, 2.5], [3.5, 4.5]]` — 모든 값에 같은 보정

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>같은 shape</td><td>1:1 계산</td><td>예측 · 정답 비교 (기본)</td></tr>
<tr><td>행렬 + scalar</td><td>모든 값에 같은 값</td><td>편향 · 정규화</td></tr>
<tr><td><code>(n, 1)</code> - <code>(n,)</code></td><td><code>(n, n)</code> 조합 · 에러 없음</td><td>하면 안 됨</td></tr>
<tr><td><code>unsqueeze(1)</code></td><td>정답을 <code>(n, 1)</code>로</td><td>1:1로 고칠 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 오류가 없다는 것은 '규칙상 계산 가능'이라는 뜻일 뿐 · 같은 위치끼리 비교했다는 보증이 아님

> ✅ **한 줄 정리** · broadcasting은 길이 1 축을 늘려 계산하므로, 비교하기 전 두 shape를 같게 만든다.

**딥러닝 코드에서는**

- `nn.MSELoss`도 `(N, 1)`과 `(N,)`이면 경고만 내고 계산함

### 4-2. `A` loss 직전 shape 확인 · `(diff ** 2).mean()` · `shape guard` <sub>원본 §4 · 셀 58 ~ 63</sub>

> 🎯 **잘못된 짝도 scalar loss 하나를 만든다면, 무엇을 근거로 옳고 그름을 판단하는가?**

💡 **쉽게 말하면**

- `mean()`은 **몇 개든 숫자 하나로 줄여 줌**.
- 그래서 loss가 scalar라는 사실은 짝이 맞았다는 증거가 아님 · 증거는 차이 텐서의 shape.

**실행 흐름**

1. 잘못된 경로 loss
2. 1:1 경로 loss
3. 실습: shape guard 통과

**먼저 예측** · 두 경로 loss는 각각 얼마인가? — 답은 코드 실행 결과 아래에

In [14]:
# 원본 셀 59: 잘못된 짝과 올바른 짝 모두 scalar loss
# 왜 · 모양이 틀려도 손실이 숫자 하나로 나와 에러 없이 틀린다는 것을 보이기 위해
wrong_difference = prediction_column - target_flat               # (4, 4)
right_difference = prediction_column - target_flat.unsqueeze(1)  # (4, 1)
wrong_scalar_loss = (wrong_difference ** 2).mean()               # 16개 평균
right_scalar_loss = (right_difference ** 2).mean()               # 4개 평균
print("잘못된 경로: difference", tuple(wrong_difference.shape),  # 짝 확인
      "→ loss", tuple(wrong_scalar_loss.shape), wrong_scalar_loss.item())
print("1:1 경로: difference", tuple(right_difference.shape),     # 짝 확인
      "→ loss", tuple(right_scalar_loss.shape), right_scalar_loss.item())


# 원본 셀 63 실습 답: shape guard 통과
# 왜 · 손실 계산 전에 모양을 assert로 검사해 조용한 버그를 막기 위해
target_attempt = target_flat.unsqueeze(1)            # 정답에 길이 1 축
if target_attempt.shape == prediction_column.shape:  # 계산 전에 비교
    print("안전한 1:1 비교 준비됨")                  # 통과 문장
else:
    # 실패 문장
    print("shape를 다시 확인함:", tuple(prediction_column.shape), tuple(target_attempt.shape))

잘못된 경로: difference (4, 4) → loss () 11.0
1:1 경로: difference (4, 1) → loss () 1.0
안전한 1:1 비교 준비됨


**결과** · 예측 정답은 11.0과 1.0 — 둘 다 scalar

- `잘못된 경로: difference (4, 4) → loss () 11.0` — 16개 조합의 평균
- `1:1 경로: difference (4, 1) → loss () 1.0` — 모든 차이 -1 → 제곱 1
- `안전한 1:1 비교 준비됨` — 원본 셀 63 실습의 답 · `unsqueeze(1)`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>loss 값만 보기</td><td>scalar면 통과처럼 보임</td><td>하지 않음</td></tr>
<tr><td>차이 텐서 shape 보기</td><td>짝 확인</td><td>loss 직전 (기본)</td></tr>
<tr><td><code>if</code> · <code>assert</code>로 guard</td><td>다르면 멈춤</td><td>학습 루프 안</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 진단 순서 · prediction shape → target shape와 축 의미 → difference shape

> ✅ **한 줄 정리** · loss 직전에는 예측 · 정답 shape와 차이 텐서의 shape를 확인해 1:1 짝을 보장한다.

**딥러닝 코드에서는**

- `assert pred.shape == y.shape` 한 줄을 학습 루프에

### 4-3. `A` 같은 위치 곱과 행렬 곱 · `*` · `@` · `torch.matmul()` <sub>원본 §4 · 셀 64 ~ 71</sub>

> 🎯 **`*`와 `@`는 무엇이 다르고, 신경망에는 왜 `@`가 필요한가?**

💡 **쉽게 말하면**

- `*`는 **같은 자리끼리 곱하기**.
- `@`는 **특성마다 곱해서 모두 더하기** · 여러 특성의 기여를 하나의 출력으로 합침.

**실행 흐름**

1. * · @ 비교
2. [3, 2] @ [2, 1]
3. matmul과 같음
4. 특성별 기여를 더해 보기

**먼저 예측** · `X @ W`의 첫 샘플 값은? (`1×2 + 2×1`) — 답은 코드 실행 결과 아래에

In [15]:
# 원본 셀 64: 같은 위치 곱 vs 곱해서 합
# 왜 · 원소별 곱과 내적의 차이를 결과 모양으로 구분하기 위해
features = torch.tensor([2.0, 3.0])                                         # 특성 2개
weights = torch.tensor([4.0, 5.0])                                          # 가중치 2개
elementwise = features * weights                                            # [2×4, 3×5]
combined = features @ weights                                               # 2×4 + 3×5
print("* 결과:", elementwise.tolist(), "shape:", tuple(elementwise.shape))  # 값 2개
print("@ 결과:", combined.item(), "shape:", tuple(combined.shape))          # 값 1개 (0차원)


# 원본 셀 67: [batch=3, in=2] @ [in=2, out=1]
# 왜 · 배치 입력과 가중치의 행렬 곱이 어떤 모양을 만드는지 확인하기 위해
X = torch.tensor([[1.0, 2.0], [2.0, 1.0], [3.0, 2.0]])                        # 샘플 3 × 특성 2
W = torch.tensor([[2.0], [1.0]])                                              # 특성 2 → 출력 1
batch_output = X @ W                                                          # (3, 1)
print("X @ W:", tuple(batch_output.shape), batch_output.squeeze(1).tolist())  # 손계산 4 · 5 · 8
print("matmul과 동일:", torch.equal(batch_output, torch.matmul(X, W)))        # 같은 연산


# 원본 셀 69: 특성별 기여를 더하면 X @ W
# 왜 · 행렬 곱이 특성별 기여의 합이라는 것을 직접 계산으로 확인하기 위해
contributions = X.unsqueeze(2) * W.unsqueeze(0)                           # (3, 2, 1) 기여
print("샘플별 특성 기여:", contributions.squeeze(2).tolist())             # [1×2, 2×1] ...
print("기여를 더한 결과:", contributions.sum(dim=1).squeeze(1).tolist())  # X @ W와 같음

* 결과: [8.0, 15.0] shape: (2,)
@ 결과: 23.0 shape: ()
X @ W: (3, 1) [4.0, 5.0, 8.0]
matmul과 동일: True
샘플별 특성 기여: [[2.0, 2.0], [4.0, 1.0], [6.0, 2.0]]
기여를 더한 결과: [4.0, 5.0, 8.0]


**결과** · 예측 정답은 `4.0` — 1×2 + 2×1 = 4

- `* 결과: [8.0, 15.0] shape: (2,)` — 2×4 · 3×5 각각
- `@ 결과: 23.0 shape: ()` — 2×4 + 3×5 = 23 · 값 하나
- `X @ W: (3, 1) [4.0, 5.0, 8.0]` — 2×2 + 1×1 = 5 · 3×2 + 2×1 = 8
- `matmul과 동일: True` — `@` = `torch.matmul`
- `기여를 더한 결과: [4.0, 5.0, 8.0]` — 기여를 더하면 X @ W

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>a * b</code></td><td>같은 위치 곱 · 모양 그대로</td><td>마스크 · 원소별 가중치</td></tr>
<tr><td><code>a @ b</code></td><td>행 × 열 곱의 합</td><td>층 계산</td></tr>
<tr><td><code>torch.matmul(a, b)</code></td><td><code>@</code>와 같음</td><td>함수 이름으로 쓸 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `*`로도 결과가 나오므로 에러가 없다고 맞는 연산인 것은 아님

> ✅ **한 줄 정리** · `*`는 같은 위치끼리 곱하고, `@`는 특성마다 곱해 더해 샘플별 출력을 만든다.

**딥러닝 코드에서는**

- `nn.Linear`의 계산 = `x @ W.T + b` (Day02)

### 4-4. `A` 결과 shape 먼저 · 두 문제 고치기 · `[batch, in] @ [in, out]` <sub>원본 §4 · 셀 72 ~ 82</sub>

> 🎯 **행렬 곱의 결과 shape를 미리 말하고, 에러가 나는 문제와 이상한 shape 문제를 고치려면?**

💡 **쉽게 말하면**

- **맞닿는 안쪽 축은 같아야 하고, 바깥 축이 결과에 남음**.
- 고칠 때는 숫자를 억지로 맞추지 않고 데이터 의미부터 확인.

**실행 흐름**

1. 네 경우의 결과 shape
2. 문제 A: 안쪽 축 3 → 2
3. 문제 B: 정답 (3,) → (3, 1)
4. 다시 실행

**먼저 예측** · `(4, 6) @ (6, 2)`의 결과는? — 답은 코드 실행 결과 아래에

In [16]:
# 원본 셀 73 · 75: 결과 shape를 먼저 말하기
# 왜 · 코드를 실행하기 전에 결과 모양을 먼저 말하는 습관을 들이기 위해
# 세 경우
matmul_cases = [(torch.ones(5, 3), torch.ones(3, 2)), (torch.ones(4, 6), torch.ones(6, 2)),
                (torch.ones(1, 5), torch.ones(5, 3)), (torch.ones(7, 4), torch.ones(3, 2))]
for left_case, right_case in matmul_cases:    # 경우마다
    try:
        output_case = left_case @ right_case  # 안쪽 축이 같아야
        # 바깥 축
        print(f"{tuple(left_case.shape)} @ {tuple(right_case.shape)} → {tuple(output_case.shape)}")
    except RuntimeError:                      # 안쪽 축이 다르면
        # 안쪽 축 4 ≠ 3
        print(f"{tuple(left_case.shape)} @ {tuple(right_case.shape)} → 실행할 수 없음")


# 원본 셀 78 · 80 실습 답: 문제 A · B를 각각 최소 수정
# 왜 · 모양 오류를 최소한으로 고치는 방법을 연습하기 위해
lab_features = torch.tensor([[1.0, 2.0], [2.0, 1.0], [3.0, 2.0]])  # 샘플 3 × 특성 2
lab_target = torch.tensor([4.0, 5.0, 8.0])                         # 정답 (3,)
weight_attempt = torch.tensor([[2.0], [1.0]])                      # A 수정: 입력 축 3 → 2
target_attempt = lab_target.unsqueeze(1)                           # B 수정: (3,) → (3, 1)
retry_prediction = lab_features @ weight_attempt                   # (3, 2) @ (2, 1)
assert retry_prediction.shape == target_attempt.shape              # 1:1 짝 확인
retry_loss = ((retry_prediction - target_attempt) ** 2).mean()     # MSE
# 정답과 같음
print("다시 실행 성공 · prediction", tuple(retry_prediction.shape), "loss:", retry_loss.item())

(5, 3) @ (3, 2) → (5, 2)
(4, 6) @ (6, 2) → (4, 2)
(1, 5) @ (5, 3) → (1, 3)
(7, 4) @ (3, 2) → 실행할 수 없음
다시 실행 성공 · prediction (3, 1) loss: 0.0


**결과** · 예측 정답은 `(4, 2)`

- `(5, 3) @ (3, 2) → (5, 2)` — 원본 셀 73
- `(1, 5) @ (5, 3) → (1, 3)` — 샘플 1개도 같은 규칙
- `(7, 4) @ (3, 2) → 실행할 수 없음` — 안쪽 4 ≠ 3
- `다시 실행 성공 · prediction (3, 1) loss: 0.0` — 원본 셀 80 실습 답 · 두 줄만 고침

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>문제 A (에러)</td><td>안쪽 축 불일치</td><td>weight 입력 축을 특성 수 2에 맞춤</td></tr>
<tr><td>문제 B (조용한 오류)</td><td><code>(3, 1) - (3,)</code> → <code>(3, 3)</code></td><td>정답에 <code>unsqueeze(1)</code></td></tr>
<tr><td>임의 reshape</td><td>숫자만 맞춤</td><td>하지 않음 · 의미 확인이 먼저</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 에러를 없애는 것이 목표가 아님 → 결과가 '샘플 3 × 출력 1' 의도와 맞는지 확인

> ✅ **한 줄 정리** · `[batch, in] @ [in, out] → [batch, out]`을 먼저 말하고, 고칠 때는 축의 의미를 확인한 뒤 최소한만 바꾼다.

**딥러닝 코드에서는**

- 에러 `mat1 and mat2 shapes cannot be multiplied (3x2 and 3x1)`의 두 안쪽 숫자를 비교

---
## 5. 예측과 틀린 정도

> **이 장의 흐름**
>
> - 4장의 계산 규칙으로 `ŷ = wx + b`를 만듦
> - 틀린 정도를 MSE 하나로 줄이고, 그 요약이 감추는 것도 확인

<img src="images/pdl2_mse.png" alt="같은 MSE 1.0 · 다른 오차 패턴" width="816">

**파이프라인 위치** · ④ 모델 → ⑤ 학습의 출발 — 예측 · 손실

### 5-1. `A` 가장 작은 모델 · `w * x + b` <sub>원본 §6 · 셀 83 ~ 88</sub>

> 🎯 **`ŷ = wx + b`에서 w와 b를 각각 1씩 올리면 예측은 어떻게 바뀌는가?**

💡 **쉽게 말하면**

- w는 **기울기 손잡이**(x가 클수록 크게), b는 **높이 손잡이**(모두 똑같이).

**실행 흐름**

1. w = b = 0의 예측
2. 기준 w = 2 · b = 0.5
3. w만 +1 · b만 +1

**먼저 예측** · 입력 `[1, 2, 4]`에서 w만 1 올리면 변화량은? — 답은 코드 실행 결과 아래에

In [17]:
# 원본 셀 84: 가장 작은 모델 ŷ = wx + b
# 왜 · 가장 작은 모델 ŷ = wx + b로 예측 계산을 직접 해 보기 위해
x_train = torch.tensor([1.0, 2.0])       # 입력
target_train = torch.tensor([3.0, 5.0])  # 정답 (규칙 2x + 1)
w = torch.tensor(0.0)                    # 아직 학습 전
b = torch.tensor(0.0)                    # 아직 학습 전
prediction = w * x_train + b             # 현재 예측
print("현재 w, b:", w.item(), b.item(), "| 현재 prediction:", prediction.tolist())  # 모두 0


# 원본 셀 86: 같은 x에서 w만 +1 · b만 +1
# 왜 · w와 b가 예측에 주는 영향을 따로 떼어 비교하기 위해
x_sensitivity = torch.tensor([1.0, 2.0, 4.0])                                     # 입력 3개
base_w, base_b = 2.0, 0.5                                                         # 기준 값
base_prediction = base_w * x_sensitivity + base_b                                 # 기준 예측
w_changed_prediction = (base_w + 1.0) * x_sensitivity + base_b                    # w만 +1
b_changed_prediction = base_w * x_sensitivity + (base_b + 1.0)                    # b만 +1
print("기준 prediction:", base_prediction.tolist())                               # 2.5 · 4.5 · 8.5
print("w만 +1 했을 때 변화:", (w_changed_prediction - base_prediction).tolist())  # x와 같음
print("b만 +1 했을 때 변화:", (b_changed_prediction - base_prediction).tolist())  # 모두 1

현재 w, b: 0.0 0.0 | 현재 prediction: [0.0, 0.0]
기준 prediction: [2.5, 4.5, 8.5]
w만 +1 했을 때 변화: [1.0, 2.0, 4.0]
b만 +1 했을 때 변화: [1.0, 1.0, 1.0]


**결과** · 예측 정답은 `[1, 2, 4]` — x에 비례

- `현재 w, b: 0.0 0.0 | 현재 prediction: [0.0, 0.0]` — 학습 전 · 처음엔 틀려도 됨
- `기준 prediction: [2.5, 4.5, 8.5]` — 2 × x + 0.5
- `w만 +1 했을 때 변화: [1.0, 2.0, 4.0]` — 변화량 = x
- `b만 +1 했을 때 변화: [1.0, 1.0, 1.0]` — 모두 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>입력 x</td><td>주어지는 값</td><td>데이터</td></tr>
<tr><td>w</td><td>x에 비례해 예측을 바꿈</td><td>입력의 영향 크기</td></tr>
<tr><td>b</td><td>모든 예측을 같은 만큼</td><td>기본값 이동</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 두 parameter의 기울기 크기가 다른 것은 자연스러움 · 중요도 차이라고 단정하지 않기

> ✅ **한 줄 정리** · w는 입력에 비례해, b는 모든 예측을 같은 만큼 바꾸며, 학습 가능성은 첫 예측이 아니라 바꿀 값 · 손실 · 방향에 있다.

**딥러닝 코드에서는**

- `nn.Linear(1, 1)`의 `weight` · `bias` = 이 w · b

### 5-2. `A` 틀린 정도를 숫자 하나로 · `MSE` <sub>원본 §7 · 셀 89 ~ 94</sub>

> 🎯 **두 예측 중 어느 쪽이 더 많이 틀렸는지 숫자 하나로 비교하려면?**

💡 **쉽게 말하면**

- MSE = 차이 → 제곱 → 평균.
- 제곱은 음수 · 양수가 상쇄되지 않게 하고 큰 오차를 더 크게 벌줌.

**실행 흐름**

1. 차이 · 제곱 · 평균
2. 예측 A · B 비교
3. RMSE와 구분

**먼저 예측** · 현재 예측 `[0, 0]`의 MSE는? — 답은 코드 실행 결과 아래에

In [18]:
# 원본 셀 91: 차이 → 제곱 → 평균 = MSE
# 왜 · MSE를 계산 순서대로 풀어 손실의 뜻을 이해하기 위해
difference = prediction - target_train         # [0-3, 0-5]
squared_error = difference ** 2                # 음수 없애기
mse = squared_error.mean()                     # 숫자 하나
print("차이:", difference.tolist())            # [-3, -5]
print("차이의 제곱:", squared_error.tolist())  # [9, 25]
print("MSE:", mse.item())                      # (9 + 25) ÷ 2


# 원본 셀 94: 두 prediction 비교
# 왜 · 손실이 낮은 예측이 더 좋은 예측이라는 것을 두 예로 확인하기 위해
prediction_a = torch.tensor([2.0, 6.0])               # 오차 -1 · +1
prediction_b = torch.tensor([1.0, 5.0])               # 오차 -2 · 0
loss_a = ((prediction_a - target_train) ** 2).mean()  # (1 + 1) ÷ 2
loss_b = ((prediction_b - target_train) ** 2).mean()  # (4 + 0) ÷ 2
print("prediction A의 MSE:", loss_a.item())           # 1.0
print("prediction B의 MSE:", loss_b.item())           # 2.0
print("RMSE(A):", loss_a.sqrt().item())               # 제곱근은 다른 이름

차이: [-3.0, -5.0]
차이의 제곱: [9.0, 25.0]
MSE: 17.0
prediction A의 MSE: 1.0
prediction B의 MSE: 2.0
RMSE(A): 1.0


**결과** · 예측 정답은 `17.0` — (9 + 25) ÷ 2

- `차이: [-3.0, -5.0]` — 0 - 3 · 0 - 5
- `MSE: 17.0` — 9 + 25 = 34, 34 ÷ 2 = 17
- `prediction A의 MSE: 1.0` — 오차 -1 · +1
- `prediction B의 MSE: 2.0` — 오차 -2 · 0 → 하나가 크게 틀려 더 큼
- `RMSE(A): 1.0` — 제곱근까지 취하면 다른 이름 (RMSE)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>MSE</td><td>제곱 평균</td><td>회귀 손실 기본</td></tr>
<tr><td>RMSE</td><td>MSE의 제곱근 · 원래 단위</td><td>설명 · 보고</td></tr>
<tr><td>차이 합</td><td>음수 · 양수가 상쇄</td><td>쓰지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - MSE 코드가 짧아도 예측 · 정답 shape가 다르면 의도하지 않은 비교를 평균냄 (4-2)

> ✅ **한 줄 정리** · MSE는 차이를 제곱해 평균 낸 숫자 하나이며, 작을수록 이 데이터에 덜 틀렸다는 뜻이다.

**딥러닝 코드에서는**

- `nn.MSELoss()(pred, y)` = `((pred - y) ** 2).mean()` (Day02)

### 5-3. `B` 같은 MSE · 다른 오차 패턴 · `squared.mean()` <sub>원본 §7 · 셀 95 ~ 98</sub>

> 🎯 **MSE가 같으면 모든 샘플이 비슷하게 틀렸다고 말할 수 있는가?**

💡 **쉽게 말하면**

- MSE는 **반 평균 점수**.
- 평균이 같아도 고르게 틀린 반과 몇 명만 크게 틀린 반이 있음.

<img src="images/pdl2_mse.png" alt="같은 MSE 1.0 · 다른 오차 패턴" width="816">

**실행 흐름**

1. 정답 0 네 개
2. 패턴 A · B의 제곱오차
3. MSE · 최대 오차

**먼저 예측** · 패턴 B의 MSE는? — 답은 코드 실행 결과 아래에

In [19]:
# 원본 셀 96: 같은 MSE · 다른 오차 패턴
# 왜 · MSE가 같아도 오차의 모양은 다를 수 있다는 한계를 보기 위해
pattern_target = torch.zeros(4)                                          # 정답 0 네 개
pattern_a = torch.tensor([-1.0, 1.0, -1.0, 1.0])                         # 고르게 틀림
root_two = torch.sqrt(torch.tensor(2.0))                                 # √2
pattern_b = torch.tensor([0.0, 0.0, -root_two.item(), root_two.item()])  # 둘만 크게 틀림

for label, pred in {"A": pattern_a, "B": pattern_b}.items():            # 두 패턴
    errors = pred - pattern_target                                      # 샘플별 오차
    squared = errors ** 2                                               # 샘플별 제곱오차
    rounded = [round(v, 3) for v in squared.tolist()]                   # 보기 좋게 반올림
    print(f"패턴 {label} | squared={rounded} | MSE={squared.mean().item():.3f}")  # 평균만 같음
    print(f"패턴 {label} | 최대 오차={errors.abs().max().item():.3f}")  # 가장 크게 틀린 정도

패턴 A | squared=[1.0, 1.0, 1.0, 1.0] | MSE=1.000
패턴 A | 최대 오차=1.000
패턴 B | squared=[0.0, 0.0, 2.0, 2.0] | MSE=1.000
패턴 B | 최대 오차=1.414


**결과** · 예측 정답은 `1.000` — A와 같음

- `패턴 A | squared=[1.0, 1.0, 1.0, 1.0] | MSE=1.000` — 모두 비슷하게 틀림
- `패턴 B | squared=[0.0, 0.0, 2.0, 2.0] | MSE=1.000` — 둘은 정확 · 둘은 크게
- `패턴 B | 최대 오차=1.414` — √2 · A의 1.0보다 큼

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>MSE 하나</td><td>평균 요약</td><td>학습 진행 비교</td></tr>
<tr><td>샘플별 오차</td><td>누가 얼마나</td><td>오류 분석</td></tr>
<tr><td>최대 오차</td><td>가장 나쁜 경우</td><td>안전이 중요한 문제</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - MSE 감소는 이 데이터의 평균 오차 감소일 뿐 · 보지 않은 데이터 성능 · 공정성까지 증명하지 않음

> ✅ **한 줄 정리** · MSE는 평균 요약이라 샘플별 오차 패턴을 감추므로, 필요하면 샘플별 오차를 따로 본다.

**딥러닝 코드에서는**

- 검증 단계에서 틀린 샘플 목록 · 클래스별 오차를 따로 확인

---
## 6. 어느 방향으로 바꿀까

> **이 장의 흐름**
>
> - 5장의 loss 17은 방향을 말해 주지 않음
> - 양옆 비교로 기울기의 뜻을 본 뒤 autograd로 자동 계산

<img src="images/dl1_slope.png" alt="위치마다 다른 기울기 (b = 0)" width="613">

**파이프라인 위치** · ⑤ 학습 — 손실 → 기울기

### 6-1. `A` 현재 위치의 국소 기울기 · `(L(w+h) - L(w-h)) / 2h` <sub>원본 §8 · 셀 99 ~ 102</sub>

> 🎯 **loss 17은 많이 틀렸다는 것만 말해 줌 · w를 키울지 줄일지는 어떻게 아는가?**

💡 **쉽게 말하면**

- 기울기는 **지금 선 자리의 비탈**.
- 양옆을 조금씩 밟아 보고 낮은 쪽으로 감.

<img src="images/dl1_slope.png" alt="위치마다 다른 기울기 (b = 0)" width="613">

**실행 흐름**

1. b = 0 고정
2. w = -0.1 · 0 · +0.1의 loss
3. 양옆 차이 ÷ 거리

**먼저 예측** · w = 0에서 변화율의 부호는? — 답은 코드 실행 결과 아래에

In [20]:
# 원본 셀 101: b = 0으로 고정하고 w 하나만 움직이기
# 왜 · 변수를 하나만 움직여 손실이 어떻게 변하는지 보고 기울기의 뜻을 잡기 위해
def loss_with_fixed_b(w_value):                                    # w 하나를 넣으면 손실
    local_prediction = w_value * x_train                           # b는 0으로 고정함
    return ((local_prediction - target_train) ** 2).mean().item()  # MSE 숫자


for nearby_w in [-0.1, 0.0, 0.1]:  # 현재 w = 0의 왼쪽 · 제자리 · 오른쪽
    print(f"w={nearby_w:+.1f} → loss={loss_with_fixed_b(nearby_w):.3f}")  # 오른쪽이 낮음

# 왜 · 양옆 차이로 구한 변화율이 기울기와 같은지 비교하기 위해
local_change_rate = (loss_with_fixed_b(0.1) - loss_with_fixed_b(-0.1)) / 0.2  # 양옆 차이 ÷ 거리
print(f"양옆 두 점으로 본 현재 근처 변화율: {local_change_rate:.3f}")         # 음수 → 키우는 방향

w=-0.1 → loss=18.325
w=+0.0 → loss=17.000
w=+0.1 → loss=15.725
양옆 두 점으로 본 현재 근처 변화율: -13.000


**결과** · 예측 정답은 음수 (-13) — 키우면 loss가 줄어듦

- `w=-0.1 → loss=18.325` — 왼쪽은 높음
- `w=+0.1 → loss=15.725` — 오른쪽이 낮음
- `양옆 두 점으로 본 현재 근처 변화율: -13.000` — 15.725 - 18.325 = -2.6, -2.6 ÷ 0.2 = -13

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>loss 값</td><td>얼마나 틀렸나</td><td>기록</td></tr>
<tr><td>기울기 부호</td><td>어느 쪽이 내리막인가</td><td>고칠 방향</td></tr>
<tr><td>기울기 크기</td><td>얼마나 가파른가</td><td>이동 크기 × 학습률</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 기울기는 목적지 좌표가 아니라 현재 위치의 정보 → 움직인 뒤에는 다시 계산

> ✅ **한 줄 정리** · 현재 위치 양옆의 loss 차이로 기울기를 재면, 부호는 고칠 방향을 크기는 가파름을 알려 준다.

**딥러닝 코드에서는**

- parameter가 수백만 개라 양옆 계산은 불가능 → autograd가 한 번에 계산 (6-2)

### 6-2. `A` requires_grad와 backward · `requires_grad` · `backward()` · `.grad` <sub>원본 §8 · 셀 103 ~ 107 · 114</sub>

> 🎯 **`loss.backward()`를 부르면 w 자체가 바뀔까, 다른 곳에 정보가 저장될까?**

💡 **쉽게 말하면**

- `requires_grad=True`는 **계산을 녹화하라는 표시**.
- `backward()`는 **녹화를 거꾸로 재생해 `.grad`에 기울기를 적음** · 값은 그대로.

**실행 흐름**

1. w · b만 추적
2. 예측 · 손실의 grad_fn
3. backward
4. 값 · .grad 확인

**먼저 예측** · backward 뒤 w는 바뀔까? — 답은 코드 실행 결과 아래에

In [21]:
# 원본 셀 105: 바꿀 값 w · b만 추적
# 왜 · 바꿀 값만 requires_grad로 추적해 기울기를 받기 위해
w = torch.tensor(0.0, requires_grad=True)                          # 기록 시작
b = torch.tensor(0.0, requires_grad=True)                          # 기록 시작
prediction = w * x_train + b                                       # 예측도 기록됨
loss = ((prediction - target_train) ** 2).mean()                   # 손실
print("loss:", loss.item())                                        # 17.0
print("prediction의 grad_fn:", type(prediction.grad_fn).__name__)  # 마지막 연산 = 더하기
print("loss의 grad_fn:", type(loss.grad_fn).__name__)              # 마지막 연산 = 평균


# 원본 셀 107: backward 전후 값 비교
# 왜 · backward는 기울기만 계산하고 값은 바꾸지 않는다는 것을 확인하기 위해
w_before_backward = w.detach().clone()                                      # 비교용 사본
loss.backward()                                                             # 기울기 계산 → .grad
print("backward 전후 w 동일:", torch.equal(w_before_backward, w.detach()))  # 값은 그대로
print("w.grad:", w.grad.item(), "| b.grad:", b.grad.item())                 # 기울기만 생김
print("x_train 추적:", x_train.requires_grad)                               # 입력은 추적 안 함

loss:

 17.0
prediction의 grad_fn: AddBackward0
loss의 grad_fn: MeanBackward0


backward 전후 w 동일: True
w.grad: -13.0 | b.grad: -8.0
x_train 추적: False


**결과** · 예측 정답은 안 바뀜 — `.grad`에 -13만 저장

- `prediction의 grad_fn: AddBackward0` — 마지막 연산(+ b)이 기록됨
- `loss의 grad_fn: MeanBackward0` — 마지막 연산(평균)
- `backward 전후 w 동일: True` — 값은 그대로
- `w.grad: -13.0 | b.grad: -8.0` — 6-1의 양옆 계산 -13과 같음
- `x_train 추적: False` — 바꿀 값이 아니라 추적 안 함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>requires_grad=True</code></td><td>계산 기록 시작</td><td>바꿀 parameter</td></tr>
<tr><td><code>loss.backward()</code></td><td>기울기만 계산 → <code>.grad</code></td><td>손실을 만든 직후</td></tr>
<tr><td><code>.grad</code></td><td>기울기 저장소</td><td>업데이트 재료</td></tr>
<tr><td><code>.grad_fn</code></td><td>이 값을 만든 연산 기록</td><td>기록되는지 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `backward()`는 업데이트가 아님 · 기울기의 음수 부호는 선악이 아니라 방향 신호

> ✅ **한 줄 정리** · `requires_grad`로 표시한 값의 계산이 기록되고, `backward()`는 기울기를 `.grad`에 저장할 뿐 값은 바꾸지 않는다.

**딥러닝 코드에서는**

- `nn.Module`의 parameter는 자동으로 `requires_grad=True` (Day02)

### 6-3. `A` 같은 loss를 두 번 backward · `backward()` · `fresh forward` <sub>원본 §8 · 셀 108 ~ 110</sub>

> 🎯 **첫 backward가 끝난 같은 loss에 다시 backward를 부르면?**

💡 **쉽게 말하면**

- backward는 **녹화를 한 번 재생하고 지움**.
- 다시 하려면 forward를 새로 해서 새 녹화를 만들어야 함.

**실행 흐름**

1. forward 한 번
2. backward 두 번 → 에러
3. zero_ → fresh forward → backward

**먼저 예측** · 두 번째 backward는 성공할까? — 답은 코드 실행 결과 아래에

In [22]:
# 원본 셀 109: 같은 loss로 backward 두 번
# 왜 · 한 번 쓴 계산 그래프로 backward를 두 번 하면 에러가 나는 이유를 보기 위해
graph_w = torch.tensor(0.0, requires_grad=True)                             # 새 w
graph_b = torch.tensor(0.0, requires_grad=True)                             # 새 b
graph_loss = ((graph_w * x_train + graph_b - target_train) ** 2).mean()     # forward 한 번
graph_loss.backward()                                                       # 첫 backward
print("첫 backward 완료, grad:", graph_w.grad.item(), graph_b.grad.item())  # -13 · -8

try:
    graph_loss.backward()      # 이미 쓴 기록을 다시 따라감
except RuntimeError as error:  # 에러를 잡아 멈추지 않게
    print("같은 loss의 두 번째 backward가 중단됨:", type(error).__name__)  # RuntimeError


# 새 forward로 fresh loss를 만들면 다시 가능
# 왜 · 매 단계 새 손실을 만들어야 하는 이유를 확인하기 위해
graph_w.grad.zero_()                                                           # 지난 기울기 비우기
graph_b.grad.zero_()                                                           # 지난 기울기 비우기
fresh_graph_loss = ((graph_w * x_train + graph_b - target_train) ** 2).mean()  # 새 기록
fresh_graph_loss.backward()                                                    # 성공
print("fresh forward 뒤 backward 완료, grad:", graph_w.grad.item(), graph_b.grad.item())  # 같은 값

첫 backward 완료, grad: -13.0 -8.0
같은 loss의 두 번째 backward가 중단됨: RuntimeError
fresh forward 뒤 backward 완료, grad: -13.0 -8.0


**결과** · 예측 정답은 실패 — `RuntimeError`

- `첫 backward 완료, grad: -13.0 -8.0` — 첫 번째는 성공
- `같은 loss의 두 번째 backward가 중단됨: RuntimeError` — 기록이 이미 해제됨
- `fresh forward 뒤 backward 완료, grad: -13.0 -8.0` — 새 기록이면 다시 가능

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>같은 loss 재사용</td><td>기록 해제 → 에러</td><td>하지 않음</td></tr>
<tr><td>매 단계 fresh forward</td><td>새 예측 · 새 손실</td><td>기본 습관</td></tr>
<tr><td><code>retain_graph=True</code></td><td>기록을 남김</td><td>Day01 범위 밖 · 특별한 경우만</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 에러를 보면 설치 문제로 단정하지 말고 '같은 loss 객체에 backward를 반복했나?'부터 확인

> ✅ **한 줄 정리** · 매 학습 단계마다 현재 parameter로 새 예측과 새 손실을 만들고 그 손실에 backward를 한다.

**딥러닝 코드에서는**

- 학습 루프 안에서 `loss = criterion(model(x), y)`를 매번 다시 계산하는 이유

### 6-4. `B` 기울기의 부호와 크기 읽기 · `±0.01` <sub>원본 §8 · 셀 111 ~ 113 · 122</sub>

> 🎯 **w.grad = -13, b.grad = -8은 각각 무엇을 말해 주는가?**

💡 **쉽게 말하면**

- 음수 부호 = **키우면 loss가 줄어드는 방향**.
- 절댓값 = 지금 위치 · 입력 크기에서의 민감도 · 중요도가 아님.

**실행 흐름**

1. w · b를 ±0.01
2. 어느 쪽이 낮은가
3. 줄어든 양 ÷ 0.01

**먼저 예측** · w를 +0.01 하면 loss는 커질까 작아질까? — 답은 코드 실행 결과 아래에

In [23]:
# 원본 셀 112: w · b를 ±0.01 움직여 부호 확인
# 왜 · 값을 조금 움직여 손실이 줄어드는 방향이 기울기 부호와 반대인지 확인하기 위해
def loss_at(w_value, b_value):                                     # w · b를 넣으면 손실
    local_prediction = w_value * x_train + b_value                 # 예측
    return ((local_prediction - target_train) ** 2).mean().item()  # MSE 숫자


epsilon = 0.01                          # 아주 조금
print("현재 loss:", loss_at(0.0, 0.0))  # 17.0
# +가 낮음
print("w를 +0.01:", round(loss_at(epsilon, 0.0), 4), "| -0.01:", round(loss_at(-epsilon, 0.0), 4))
# +가 낮음
print("b를 +0.01:", round(loss_at(0.0, epsilon), 4), "| -0.01:", round(loss_at(0.0, -epsilon), 4))


# 줄어든 양 ÷ 0.01 ≈ |기울기|
# 왜 · 작게 움직인 변화율이 기울기 크기와 맞는지 숫자로 확인하기 위해
print("w 쪽 변화율:", round((loss_at(epsilon, 0.0) - 17.0) / epsilon, 2))  # ≈ -13
print("b 쪽 변화율:", round((loss_at(0.0, epsilon) - 17.0) / epsilon, 2))  # ≈ -8

현재 loss: 17.0
w를 +0.01: 16.8703 | -0.01: 17.1302
b를 +0.01: 16.9201 | -0.01: 17.0801
w 쪽 변화율: -12.97
b 쪽 변화율: -7.99


**결과** · 예측 정답은 작아짐 — 17 → 16.8703

- `w를 +0.01: 16.8703 | -0.01: 17.1302` — 키우는 쪽이 낮음
- `b를 +0.01: 16.9201 | -0.01: 17.0801` — b도 키우는 쪽
- `w 쪽 변화율: -12.97` — ≈ w.grad -13
- `b 쪽 변화율: -7.99` — ≈ b.grad -8

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>부호</td><td>방향</td><td>업데이트 식에서 빼면 키우게 됨</td></tr>
<tr><td>크기</td><td>현재 민감도</td><td>이동 크기에 영향</td></tr>
<tr><td><code>|w.grad| &gt; |b.grad|</code></td><td>입력 크기 · 단위 때문일 수 있음</td><td>중요도로 해석하지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 한 위치의 기울기를 모든 위치로 일반화하지 않음

> ✅ **한 줄 정리** · 기울기의 부호는 방향, 크기는 현재 위치의 민감도이며, 업데이트 식에서 빼면 음수 기울기의 parameter는 커진다.

**딥러닝 코드에서는**

- `w -= lr * w.grad` → `-13`이면 w가 `+1.3`만큼 (lr 0.1)

### 6-5. `A` 기울기 누적과 reset · `grad.zero_()` <sub>원본 §8 · 셀 115 ~ 121</sub>

> 🎯 **`.grad`를 비우지 않고 다시 backward 하면 기울기는 덮어써질까, 더해질까?**

💡 **쉽게 말하면**

- `.grad`는 **덮어쓰는 칠판이 아니라 쌓이는 저금통**.
- 한 단계에 한 loss의 기울기만 쓰려면 매번 비움.

**실행 흐름**

1. reset 없이 두 번
2. 매번 reset
3. reset 뒤 parameter 확인

**먼저 예측** · reset 없이 두 번째 w.grad는? — 답은 코드 실행 결과 아래에

In [24]:
# 원본 셀 118: reset을 빼먹으면 · 매번 하면
# 왜 · 기울기를 지우지 않으면 누적되는 함정을 직접 비교하기 위해
def collect_grad_trace(reset_each_time):             # reset 여부만 바꾼 실험
    trace_w = torch.tensor(0.0, requires_grad=True)  # 새 w
    trace_b = torch.tensor(0.0, requires_grad=True)  # 새 b
    records = []                                     # (반복, w.grad, b.grad)
    for repeat in range(1, 3):                       # 같은 현재 값에서 두 번
        trace_loss = ((trace_w * x_train + trace_b - target_train) ** 2).mean()  # fresh loss
        trace_loss.backward()                        # 기울기 → .grad에 더해짐
        records.append((repeat, trace_w.grad.item(), trace_b.grad.item()))  # 기록
        if reset_each_time:                          # 매번 비우기
            trace_w.grad.zero_()                     # w.grad = 0
            trace_b.grad.zero_()                     # b.grad = 0
    return records                                   # 두 번의 기록


print("reset 누락:", collect_grad_trace(False))  # 두 번째가 두 배
print("매번 reset:", collect_grad_trace(True))   # 두 번 모두 같음


# 원본 셀 121: reset은 기울기만 0 · parameter는 그대로
# 왜 · zero_()는 기울기만 지우고 값은 그대로 둔다는 것을 확인하기 위해
w.grad.zero_()  # 6-2의 w
b.grad.zero_()  # 6-2의 b
# 0 · 0 | 0 · 0
print("reset 뒤:", w.grad.item(), b.grad.item(), "| parameter는 여전히:", w.item(), b.item())

reset 누락: [(1, -13.0, -8.0), (2, -26.0, -16.0)]
매번 reset: [(1, -13.0, -8.0), (2, -13.0, -8.0)]
reset 뒤: 0.0 0.0 | parameter는 여전히: 0.0 0.0


**결과** · 예측 정답은 `-26.0` — 더해짐

- `reset 누락: [(1, -13.0, -8.0), (2, -26.0, -16.0)]` — 같은 기울기가 한 번 더 더해짐
- `매번 reset: [(1, -13.0, -8.0), (2, -13.0, -8.0)]` — 매번 새로
- `reset 뒤: 0.0 0.0 | parameter는 여전히: 0.0 0.0` — 기울기만 0 · 값은 그대로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>reset 누락</td><td>기울기 누적</td><td>일부러 모을 때만</td></tr>
<tr><td><code>grad.zero_()</code></td><td>기울기만 0</td><td>직접 만든 루프 (이 권)</td></tr>
<tr><td><code>optimizer.zero_grad()</code></td><td>모든 parameter 한 번에</td><td>Day02</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 누적 자체가 버그는 아님 · 오늘 계약(한 단계 = 한 loss)에서는 reset 누락이 오류

> ✅ **한 줄 정리** · 기울기는 `.grad`에 누적되므로 다음 단계 전에 `grad.zero_()`로 비우고, parameter 값은 그대로 남는다.

**딥러닝 코드에서는**

- `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`

---
## 7. 실제로 바꾸고 반복하기

> **이 장의 흐름**
>
> - 6장의 기울기로 값을 실제로 바꿈
> - 한 단계를 반복하고, 보폭(학습률)만 바꿔 경로를 비교

<img src="images/dl1_cycle.png" alt="학습 한 바퀴: 예측 → 손실 → backward → step → 비우기" width="666">

**파이프라인 위치** · ⑤ 학습 — 업데이트 · 반복 · 학습률

### 7-1. `A` 한 학습 단계 · `torch.no_grad()` · `w -= lr * w.grad` <sub>원본 §9 · 셀 123 ~ 130</sub>

> 🎯 **기울기로 parameter가 실제로 바뀌는 순간은 언제인가?**

💡 **쉽게 말하면**

- backward는 **지도를 그리는 것**, 업데이트는 **실제로 걷는 것**.
- 걷는 동작은 녹화하지 않게 `no_grad` 안에서.

<img src="images/dl1_cycle.png" alt="학습 한 바퀴: 예측 → 손실 → backward → step → 비우기" width="666">

**실행 흐름**

1. 예측 → 손실 → backward
2. no_grad 안에서 업데이트
3. reset
4. 새 예측 · 새 손실
5. 손계산표와 대조

**먼저 예측** · 학습률 0.1 · w.grad = -13이면 새 w는? — 답은 코드 실행 결과 아래에

In [25]:
# 원본 셀 127: 한 학습 단계 (학습률 0.1)
# 왜 · 한 학습 단계를 손계산과 같은 순서로 실행해 결과를 맞춰 보기 위해
w = torch.tensor(0.0, requires_grad=True)  # 새로 시작
b = torch.tensor(0.0, requires_grad=True)  # 새로 시작
step_size = 0.1                            # 학습률

# 왜 · 갱신 전 손실을 기록해 한 단계 뒤 줄었는지 비교하기 위해
prediction_before = w * x_train + b                                                  # 1. 예측
loss_before = ((prediction_before - target_train) ** 2).mean()                       # 2. 손실
loss_before.backward()                                                               # 3. 기울기
print(f"before: w={w.item():.3f}, b={b.item():.3f}, loss={loss_before.item():.3f}")  # 0 · 0 · 17
print(f"gradient: w.grad={w.grad.item():.3f}, b.grad={b.grad.item():.3f}")           # -13 · -8

with torch.no_grad():        # 4. 업데이트는 기록하지 않음
    w -= step_size * w.grad  # 0 - 0.1 × (-13)
    b -= step_size * b.grad  # 0 - 0.1 × (-8)
w.grad.zero_()               # 5. 다음 단계를 위해 비움
b.grad.zero_()               # 5. 다음 단계를 위해 비움

# 왜 · 갱신 후 새 예측으로 손실을 다시 계산하기 위해
prediction_after = w * x_train + b                                 # 6. 바뀐 값으로 새 예측
loss_after = ((prediction_after - target_train) ** 2).mean()       # 새 손실
print(f"after: w={w.item():.3f}, b={b.item():.3f}")                # 1.3 · 0.8
new_prediction = [round(v, 3) for v in prediction_after.tolist()]  # 보기 좋게 반올림
print("new prediction:", new_prediction, f"| new loss: {loss_after.item():.3f}")  # 17 → 1.685


# 원본 셀 125 손계산 기록표 답
# 왜 · 코드 결과와 손계산 결과가 같은지 검증하기 위해
manual_scaffold = {"new_w": 1.3, "new_b": 0.8,  # 0 - 0.1 × (-13) · 0 - 0.1 × (-8)
                   "prediction_1": 2.1, "prediction_2": 3.4, "new_loss": 1.685}
# 실행 값과 대조
print("손계산과 같음:", abs(manual_scaffold["new_loss"] - loss_after.item()) < 1e-6)

before: w=0.000, b=0.000, loss=17.000
gradient: w.grad=-13.000, b.grad=-8.000
after: w=1.300, b=0.800
new prediction: [2.1, 3.4] | new loss: 1.685
손계산과 같음: True


**결과** · 예측 정답은 `1.3` — 0 - 0.1 × (-13) = 1.3

- `gradient: w.grad=-13.000, b.grad=-8.000` — 6-2와 같은 기울기
- `after: w=1.300, b=0.800` — 0.1 × 13 = 1.3 · 0.1 × 8 = 0.8
- `new prediction: [2.1, 3.4] | new loss: 1.685` — 1.3 + 0.8 = 2.1 · 17 → 1.685
- `손계산과 같음: True` — 원본 손계산 기록표(빈칸)의 답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>backward()</code></td><td>기울기만 계산</td><td>업데이트 전</td></tr>
<tr><td><code>no_grad</code> 안 <code>-=</code></td><td>값이 실제로 바뀜</td><td>업데이트</td></tr>
<tr><td><code>grad.zero_()</code></td><td>다음 단계 준비</td><td>업데이트 뒤</td></tr>
<tr><td>fresh 예측 · 손실</td><td>바뀐 값의 성적</td><td>효과 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 업데이트 식은 빼기 → `+=`로 쓰면 loss가 커지는 쪽으로 감

> ✅ **한 줄 정리** · parameter는 `backward()`가 아니라 `no_grad` 안의 `w -= lr * w.grad`에서 바뀌고, 바뀐 값으로 다시 예측해 확인한다.

**딥러닝 코드에서는**

- `optimizer.step()`이 같은 업데이트를 모든 parameter에 대신 해 줌

### 7-2. `A` no_grad · detach · item · `torch.no_grad()` · `.detach()` · `.item()` <sub>원본 §9 · 셀 131 ~ 133</sub>

> 🎯 **세 가지 모두 '기록에서 빠진다'는데 책임은 어떻게 다른가?**

💡 **쉽게 말하면**

- `no_grad`는 **블록 안 계산 전체의 녹화 끄기**.
- `detach`는 **텐서 하나만 떼어 낸 사본**, `item`은 **Python 숫자로 꺼내기**.

**실행 흐름**

1. 기록되는 값
2. detach · item · no_grad 결과
3. item에 backward가 있나

**먼저 예측** · `item()` 결과에 `.backward()`를 부를 수 있을까? — 답은 코드 실행 결과 아래에

In [26]:
# 원본 셀 132: no_grad · detach · item의 책임
# 왜 · no_grad · detach · item이 각각 무엇을 끊는지 구분하기 위해
demo_value = torch.tensor(3.0, requires_grad=True)  # 기록되는 값
tracked_result = demo_value * 2                     # 기록됨
detached_result = tracked_result.detach()           # 텐서 하나를 기록에서 떼어 냄
python_number = tracked_result.item()               # Python 숫자
with torch.no_grad():                               # 블록 안 계산 전부 기록 안 함
    untracked_result = demo_value * 2               # 기록 없음

print("tracked:", type(tracked_result).__name__, tracked_result.requires_grad)     # Tensor True
print("detached:", type(detached_result).__name__, detached_result.requires_grad)  # Tensor False
print("item:", type(python_number).__name__)                                       # float
# Tensor False
print("no_grad 안 결과:", type(untracked_result).__name__, untracked_result.requires_grad)


# item() 결과에는 backward를 부를 수 없음 (원본 셀 131 질문)
# 왜 · 숫자로 꺼낸 값은 그래프와 끊긴다는 것을 확인하기 위해
print("item에 backward가 있나?", hasattr(python_number, "backward"))  # False

tracked: Tensor True
detached: Tensor False
item: float
no_grad 안 결과: Tensor False
item에 backward가 있나? False


**결과** · 예측 정답은 없음 — Python 숫자라 텐서 기능이 없음

- `tracked: Tensor True` — 기록됨
- `detached: Tensor False` — 텐서지만 기록 없음
- `item: float` — Python 숫자
- `no_grad 안 결과: Tensor False` — 블록 안 계산은 기록 안 됨
- `item에 backward가 있나? False` — 원본 질문(item에 backward?)의 답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>with torch.no_grad():</code></td><td>블록 전체 기록 끔</td><td>업데이트 · 평가</td></tr>
<tr><td><code>.detach()</code></td><td>텐서 하나 사본 (값 공유)</td><td>출력 · 저장용 텐서</td></tr>
<tr><td><code>.item()</code></td><td>Python 숫자</td><td>출력 · 기록</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `detach()` · `item()`을 loss 계산 중간에 넣으면 기울기 경로가 끊김

> ✅ **한 줄 정리** · `no_grad`는 블록 전체, `detach`는 텐서 하나를 기록에서 빼며, `item`은 숫자로 꺼낼 뿐 셋 다 업데이트를 대신하지 않는다.

**딥러닝 코드에서는**

- 평가 `with torch.no_grad():` · 기록 `loss.item()`

### 7-3. `A` 반복 가능한 학습 구조 · `for step in range()` <sub>원본 §9 · 셀 134 · 135 · 139 ~ 144</sub>

> 🎯 **한 단계를 반복하면 어떻게 되고, 세 책임이 빠진 loop는 어디서 깨지는가?**

💡 **쉽게 말하면**

- `계산 → 방향 → 변경 → 정리 → 재확인`을 **같은 순서로 반복**.
- 업데이트 · reset · fresh forward가 빠지면 같은 loss를 다시 쓰다 에러가 남.

**실행 흐름**

1. 4단계 반복
2. 망가진 loop 재현
3. 실습: 한 단계의 순서

**먼저 예측** · 4단계 뒤 w는 정답 2에 가까워질까? — 답은 코드 실행 결과 아래에

In [27]:
# 원본 셀 135: 같은 순서를 4번 반복
# 왜 · 한 단계를 반복하면 손실이 계속 줄어드는지 확인하기 위해
w_loop = torch.tensor(0.0, requires_grad=True)                             # 시작 w
b_loop = torch.tensor(0.0, requires_grad=True)                             # 시작 b
for step in range(1, 5):                                                   # 4단계
    loss_loop = ((w_loop * x_train + b_loop - target_train) ** 2).mean()   # 예측 → 손실
    loss_loop.backward()                                                   # 기울기
    with torch.no_grad():                                                  # 업데이트
        w_loop -= 0.1 * w_loop.grad                                        # w 이동
        b_loop -= 0.1 * b_loop.grad                                        # b 이동
    w_loop.grad.zero_()                                                    # 비우기
    b_loop.grad.zero_()                                                    # 비우기
    fresh_loss = ((w_loop * x_train + b_loop - target_train) ** 2).mean()  # 바뀐 값의 성적
    print(f"step {step}: w={w_loop.item():.4f}, b={b_loop.item():.4f},"    # 바뀐 값
          f" fresh loss={fresh_loss.item():.6f}")


# 원본 셀 140: 세 책임이 빠진 loop → 같은 loss 재사용 에러
# 왜 · 학습 루프의 세 책임 중 하나라도 빠지면 생기는 에러를 재현하기 위해
broken_w = torch.tensor(0.0, requires_grad=True)                 # 새 w
broken_loss = ((broken_w * x_train - target_train) ** 2).mean()  # forward는 한 번만
broken_loss.backward()                                           # 1단계는 성공
try:
    broken_loss.backward()                                       # 2단계: 같은 loss를 다시
except RuntimeError as error:                                    # 에러를 잡아 멈추지 않게
    print("broken step 2:", type(error).__name__, "| w 그대로:", broken_w.item())  # 값도 안 바뀜


# 원본 셀 144 답: 한 단계의 실제 순서
# 왜 · 학습 한 단계의 실제 순서를 코드와 맞춰 정리하기 위해
student_order = ["prediction과 loss", "backward", "parameter update",  # 아직 없는 기울기는 못 씀
                 "gradient reset", "fresh prediction과 new loss"]
print(" → ".join(student_order))                                       # 한 줄로

step 1: w=1.3000, b=0.8000, fresh loss=1.685000
step 2: w=1.7100, b=1.0500, fresh loss=0.169250
step 3: w=1.8400, b=1.1270, fresh loss=0.019169
step 4: w=1.8819, b=1.1496, fresh loss=0.004246
broken step 2: RuntimeError | w 그대로: 0.0
prediction과 loss → backward → parameter update → gradient reset → fresh prediction과 new loss


**결과** · 예측 정답은 가까워짐 — w 1.8819 · b 1.1496

- `step 1: w=1.3000, b=0.8000, fresh loss=1.685000` — 7-1과 같은 첫 단계
- `step 4: w=1.8819, b=1.1496, fresh loss=0.004246` — w → 2 · b → 1에 접근
- `broken step 2: RuntimeError | w 그대로: 0.0` — 업데이트 · fresh forward 없음
- `prediction과 loss → backward → parameter update → gradient reset → fresh prediction과 new loss` — 원본 직접 확인하기(순서 복원)의 답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>완성된 loop</td><td>다섯 책임 모두</td><td>학습</td></tr>
<tr><td>update 빠짐</td><td>값이 안 바뀜</td><td>버그</td></tr>
<tr><td>reset 빠짐</td><td>기울기 누적</td><td>버그</td></tr>
<tr><td>fresh forward 빠짐</td><td>같은 loss 재사용 에러</td><td>버그</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 셀 142의 세 표식(fresh forward · update · reset)은 이 세 가지가 빠진 자리를 가리킴

> ✅ **한 줄 정리** · 학습은 예측 → 손실 → backward → 업데이트 → reset을 반복하는 것이며, 하나라도 빠지면 학습이 멈추거나 에러가 난다.

**딥러닝 코드에서는**

- Day02의 `zero_grad → forward → loss → backward → step`이 같은 다섯 책임

### 7-4. `A` 학습률이 바꾸는 경로 · `lr` <sub>원본 §10 · 셀 145 ~ 154</sub>

> 🎯 **같은 데이터 · 시작값에서 한 번에 움직이는 크기만 바꾸면 무엇이 달라지는가?**

💡 **쉽게 말하면**

- 학습률은 **보폭**.
- 작으면 느리고, 크면 바닥을 건너뛰어 반대편 더 높은 곳으로.

<img src="images/pdl2_lr.png" alt="학습률 0.001 · 0.1 · 0.5의 loss 경로 (로그 눈금)" width="630">

**실행 흐름**

1. 실험 A · B · C (0.001 · 0.1 · 0.5)
2. 8단계 경로
3. 중간 0.2 · 0.3 (12단계)
4. 단조 감소 · 증가 판정

**먼저 예측** · 0.5에서 셋째 기록의 loss는 처음보다 작을까? — 답은 코드 실행 결과 아래에

In [28]:
# 원본 셀 146: 학습률만 바꾸는 실험 함수 (같은 데이터 · 같은 시작값)
# 왜 · 학습률만 바꾸고 나머지는 같게 해 공정하게 비교하기 위해
def run_step_size_trial(step_size, steps=8):                    # 학습률만 받는 실험
    trial_w = torch.tensor(0.0, requires_grad=True)             # 실험마다 새로
    trial_b = torch.tensor(0.0, requires_grad=True)             # 실험마다 새로
    history = []                                                # 업데이트 전 loss
    for _ in range(steps):                                      # steps번
        trial_loss = ((trial_w * x_train + trial_b - target_train) ** 2).mean()  # 손실
        history.append(trial_loss.item())                       # 기록
        trial_loss.backward()                                   # 기울기
        with torch.no_grad():                                   # 업데이트
            trial_w -= step_size * trial_w.grad                 # 이동 = 학습률 × 기울기
            trial_b -= step_size * trial_b.grad                 # b도 같은 규칙
        trial_w.grad.zero_()                                    # 비우기
        trial_b.grad.zero_()                                    # 비우기
    final_loss = ((trial_w * x_train + trial_b - target_train) ** 2).mean().item()  # 마지막 성적
    return history, trial_w.item(), trial_b.item(), final_loss  # 경로 · 도착점


# 원본 셀 148: 실험 A · B · C (8단계)
# 왜 · 학습률이 너무 작거나 크면 어떻게 되는지 세 실험으로 비교하기 위해
for label, size in {"실험 A": 0.001, "실험 B": 0.1, "실험 C": 0.5}.items():  # 학습률 3개
    history, final_w, final_b, final_loss = run_step_size_trial(size)        # 같은 조건
    print(f"{label} · 크기={size} | 처음 {history[0]:.3f}"                   # 경로 요약
          f" → 셋째 {history[2]:.3f} → 끝 {final_loss:.6g}")


# 원본 셀 150: 중간 학습률 0.2 · 0.3 (12단계)
# 왜 · 중간 학습률에서의 결과를 더 촘촘히 확인하기 위해
for size in [0.2, 0.3]:                                       # 두 후보
    history = run_step_size_trial(size, steps=12)[0]          # 경로만
    down = all(n <= p for p, n in zip(history, history[1:]))  # 매번 줄었나
    up = all(n >= p for p, n in zip(history, history[1:]))    # 매번 커졌나
    # 세 점
    print(f"lr={size}: 처음 {history[0]:.1f} · 둘째 {history[1]:.4g} · 마지막 {history[-1]:.4g}"
          f" | 단조 감소={down} · 단조 증가={up}")

실험 A · 크기=0.001 | 처음 17.000 → 셋째 16.539 → 끝 15.2288
실험 B · 크기=0.1 | 처음 17.000 → 셋째 0.169 → 끝 0.0023254
실험 C · 크기=0.5 | 처음 17.000 → 셋째 589.781 → 끝 2.46397e+07
lr=0.2: 처음 17.0 · 둘째 2.34 · 마지막 0.001533 | 단조 감소=True · 단조 증가=False
lr=0.3: 처음 17.0 · 둘째 18.97 · 마지막 56.63 | 단조 감소=False · 단조 증가=True


**결과** · 예측 정답은 훨씬 큼 — 17 → 589.781

- `실험 A · 크기=0.001 | 처음 17.000 → 셋째 16.539 → 끝 15.2288` — 느림
- `실험 B · 크기=0.1 | 처음 17.000 → 셋째 0.169 → 끝 0.0023254` — 빠르게 감소
- `실험 C · 크기=0.5 | 처음 17.000 → 셋째 589.781 → 끝 2.46397e+07` — 발산
- `lr=0.2: 처음 17.0 · 둘째 2.34 · 마지막 0.001533 | 단조 감소=True · 단조 증가=False` — 안정적 감소
- `lr=0.3: 처음 17.0 · 둘째 18.97 · 마지막 56.63 | 단조 감소=False · 단조 증가=True` — 원본 설명대로 매 기록 증가 (발산)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>너무 작음 (0.001)</td><td>같은 방향이지만 진전이 느림</td><td>loss가 거의 안 줄면 키움</td></tr>
<tr><td>적절 (0.1 · 0.2)</td><td>빠르게 감소</td><td>이 데이터에서</td></tr>
<tr><td>너무 큼 (0.3 · 0.5)</td><td>지나쳐서 커짐</td><td>loss가 커지면 줄임</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 작은 실험이 모든 모델의 최적 학습률을 증명하지 않음 · 경로 전체를 보고 판단

> ✅ **한 줄 정리** · 학습률은 업데이트 크기를 정하므로 작으면 느리고 크면 발산하며, 마지막 loss가 아니라 경로로 판단한다.

**딥러닝 코드에서는**

- `optim.SGD(lr=0.1)` · loss가 `nan`이면 학습률부터 낮춤

---
## 8. 종합 실습 · 새 숫자에 같은 학습 구조 적용

> **이 장의 흐름**
>
> - 원본 §9의 '다른 숫자에 같은 학습 구조 적용'(셀 136 ~ 138)과 '전체 흐름 코드 읽기'(셀 155 ~ 157)
> - 손계산 → 학습률 선택 → 경로 비교 → 이름이 바뀐 코드에서 역할 찾기

<img src="images/pdl2_practice.png" alt="학습률 0.03 · 0.05의 3단계 fresh loss" width="531">

**파이프라인 위치** · ④ 모델 → ⑤ 학습 → ⑥ 확인

> 🎯 **새 데이터 x = [1, 3] · 정답 [2, 6]에서 학습률 0.03과 0.05 중 3단계 뒤 loss가 더 작은 쪽은?**

💡 **쉽게 말하면**

- 같은 요리법(학습 구조)을 **새 재료**(새 숫자)에 써 보는 것.
- 재료 이름이 바뀌어도(`slope` · `score`) 손질 순서는 같음.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>첫 예측 · 첫 loss 손계산 후 실행값과 대조</td><td>5-1 · 5-2</td></tr>
<tr><td>2</td><td>학습률 0.03 · 0.05로 3단계씩</td><td>7-3 · 7-4</td></tr>
<tr><td>3</td><td>fresh loss 경로 · w · b 이동 비교</td><td>7-4</td></tr>
<tr><td>4</td><td>이름이 바뀐 코드에서 여섯 역할 찾기</td><td>7-3</td></tr>
</tbody></table>

**먼저 예측** · 첫 loss는? 0.03 · 0.05 중 3단계 뒤 loss가 작은 쪽은? — 답은 코드 실행 결과 아래에

In [29]:
# 원본 셀 137 답: 새 데이터 x = [1, 3] · 정답 [2, 6] · 시작 w = b = 0.5
# 왜 · 새 데이터에도 같은 학습 순서가 그대로 쓰이는지 손계산과 맞춰 보기 위해
transfer_x = torch.tensor([1.0, 3.0])                           # 입력
transfer_target = torch.tensor([2.0, 6.0])                      # 정답 (규칙 2x)
first_prediction_attempt, first_loss_attempt = [1.0, 2.0], 8.5  # 손계산 (예측 두 개 · 첫 손실)

# 왜 · 코드의 첫 예측 · 손실이 손계산과 같은지 확인하기 위해
actual_first_prediction = 0.5 * transfer_x + 0.5                               # 실제 첫 예측
actual_first_loss = ((actual_first_prediction - transfer_target) ** 2).mean()  # 실제 첫 손실
# 같음
print("첫 prediction 손계산 / 실제:", first_prediction_attempt, actual_first_prediction.tolist())
print("첫 loss 손계산 / 실제:", first_loss_attempt, actual_first_loss.item())  # 같음

첫 prediction 손계산 / 실제: [1.0, 2.0] [1.0, 2.0]
첫 loss 손계산 / 실제: 8.5 8.5


**실행 흐름**

1. 같은 데이터 · 같은 시작값으로 학습 함수 준비
2. 학습률 0.03 · 0.05로 3단계씩 학습
3. 학습률별 w · b · 손실 출력

In [30]:
# 학습률 0.03 · 0.05 두 가지로 3단계씩 (원본은 하나만 고르게 함)
# 왜 · 원본이 하나만 고르게 한 학습률을 둘 다 시험해 선택 근거를 보이기 위해
def transfer_run(step_size, steps=3):                                      # 학습률만 바꿔 3단계
    t_w = torch.tensor(0.5, requires_grad=True)                            # 시작 w
    t_b = torch.tensor(0.5, requires_grad=True)                            # 시작 b
    path = []                                                              # 단계별 fresh loss
    for _ in range(steps):                                                 # 3단계
        t_loss = ((t_w * transfer_x + t_b - transfer_target) ** 2).mean()  # 손실
        t_loss.backward()                                                  # 기울기
        with torch.no_grad():                                              # 업데이트
            t_w -= step_size * t_w.grad                                    # w 이동
            t_b -= step_size * t_b.grad                                    # b 이동
        t_w.grad.zero_()                                                   # 비우기
        t_b.grad.zero_()                                                   # 비우기
        # fresh loss
        path.append(round(((t_w * transfer_x + t_b - transfer_target) ** 2).mean().item(), 4))
    return round(t_w.item(), 4), round(t_b.item(), 4), path                # 도착점 · 경로


for step_size_choice in [0.03, 0.05]:  # 원본이 제시한 두 후보
    print(f"lr {step_size_choice}: w, b, fresh loss 경로 =", transfer_run(step_size_choice))  # 비교

lr 0.03: w, b, fresh loss 경로 = (1.3122, 0.8021, [3.697, 1.6638, 0.8019])
lr 0.05: w, b, fresh loss 경로 = (1.5435, 0.8755, [1.625, 0.4237, 0.2098])


**실행 흐름**

1. 이름이 바뀐 변수로 예측 → 손실 → backward → 갱신 → reset 한 단계 실행
2. 새 손실(check_score)로 줄었는지 비교
3. 역할과 코드 줄 연결 개수 출력

In [31]:
# 원본 셀 155 · 156 답: 이름이 바뀐 코드에서 역할 찾기
# 왜 · 이름이 바뀐 코드에서도 같은 역할을 찾아 읽는 힘을 기르기 위해
features_new = torch.tensor([1.0, 2.0])                                    # 입력
labels_new = torch.tensor([3.0, 5.0])                                      # 정답
slope = torch.tensor(0.0, requires_grad=True)                              # parameter w
offset = torch.tensor(0.0, requires_grad=True)                             # parameter b
estimate = slope * features_new + offset                                   # prediction
score = ((estimate - labels_new) ** 2).mean()                              # loss
score.backward()                                                           # backward
with torch.no_grad():                                                      # update
    slope -= 0.1 * slope.grad                                              # w 이동
    offset -= 0.1 * offset.grad                                            # b 이동
slope.grad.zero_()                                                         # reset
offset.grad.zero_()                                                        # reset
check_estimate = slope * features_new + offset                             # fresh evidence
check_score = ((check_estimate - labels_new) ** 2).mean()                  # 새 손실
print(f"score {score.item():.3f} → check_score {check_score.item():.3f}")  # 7-1과 같은 숫자
# 역할 → 줄
code_roles = {"prediction": "estimate", "loss": "score", "backward": "score.backward()",
              "update": "-=", "reset": "zero_()", "fresh_evidence": "check_score"}
print(f"역할 연결: {sum(bool(v) for v in code_roles.values())}/{len(code_roles)}개 작성함")  # 6/6

score 17.000 → check_score 1.685
역할 연결: 6/6개 작성함


**결과** · 예측 정답은 첫 loss `8.5` · 0.05가 더 작음 (0.2098 vs 0.8019)

- `첫 prediction 손계산 / 실제: [1.0, 2.0] [1.0, 2.0]` — 0.5 × 1 + 0.5 = 1 · 0.5 × 3 + 0.5 = 2
- `첫 loss 손계산 / 실제: 8.5 8.5` — 오차 -1 · -4 → 1 + 16 = 17, 17 ÷ 2 = 8.5
- `lr 0.03: w, b, fresh loss 경로 = (1.3122, 0.8021, [3.697, 1.6638, 0.8019])` — 작은 보폭 · 꾸준히 감소
- `lr 0.05: w, b, fresh loss 경로 = (1.5435, 0.8755, [1.625, 0.4237, 0.2098])` — 같은 3단계에 더 많이 감소
- `score 17.000 → check_score 1.685` — 이름만 바뀐 7-1 · 숫자가 같음
- `역할 연결: 6/6개 작성함` — 원본 역할 연결 기록표의 답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>lr 0.03</td><td>w 0.5 → 1.3122</td><td>안전하지만 느림</td></tr>
<tr><td>lr 0.05</td><td>w 0.5 → 1.5435</td><td>이 데이터 · 3단계에서 더 빠름</td></tr>
<tr><td>마지막 loss만 보기</td><td>도착 여부</td><td>빠른 확인</td></tr>
<tr><td>경로 · w · b 이동까지 보기</td><td>어떻게 가까워졌나</td><td>학습률 선택 근거</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 작은 학습 데이터의 loss 감소만으로 보지 않은 데이터 성능을 주장하지 않음

> ✅ **한 줄 정리** · 숫자와 이름이 바뀌어도 예측 → 손실 → backward → 업데이트 → reset 순서를 찾으면 같은 학습 구조로 읽고 실행할 수 있다.

**딥러닝 코드에서는**

- Day02에서는 이 구조를 `nn.Module` · `optimizer` · `DataLoader`가 나눠 맡음 (파일별 3)

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 텐서 읽기</td><td><code>mini_images</code> = 샘플 2 · 채널 3 · 높이 2 · 너비 2 · 범주 번호는 int64</td><td>1장</td></tr>
<tr><td>2. 꺼내기 · 모양</td><td>습도 <code>(4,)</code> · <code>[1]</code><code>(3,)</code> vs <code>[1:2]</code><code>(1, 3)</code> · reshape ≠ <code>.T</code></td><td>2 · 3장</td></tr>
<tr><td>3. shape 계약</td><td><code>[4, 1] - [4]</code> → <code>(4, 4)</code> · loss 11.0 vs 1.0 · <code>(4, 6) @ (6, 2)</code> → <code>(4, 2)</code></td><td>4장</td></tr>
<tr><td>4. 손실 · 기울기</td><td>MSE 17 · 양옆 기울기 -13 = <code>w.grad</code> · 누적 -26</td><td>5 · 6장</td></tr>
<tr><td>5. 업데이트 · 학습률</td><td>w 0 → 1.3 · 4단계 뒤 1.8819 · 0.3은 매 기록 증가</td><td>7 · 8장</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>텐서 만들기</td><td>값 · 틀 · 수열 · 난수 + seed</td><td><code>torch.manual_seed(42)</code></td></tr>
<tr><td><code>A</code></td><td>shape · dtype</td><td>축 의미까지 · 범주는 int64</td><td><code>x.shape</code> · <code>y.dtype</code></td></tr>
<tr><td><code>A</code></td><td>선택</td><td>정수는 축 사라짐 · 범위는 유지</td><td><code>x[0:1]</code></td></tr>
<tr><td><code>A</code></td><td>모양</td><td>reshape는 원소 수 · <code>.T</code>는 축 교환</td><td><code>x.reshape(n, -1)</code></td></tr>
<tr><td><code>A</code></td><td>cat · stack</td><td>기존 축 · 새 축</td><td>배치 = stack</td></tr>
<tr><td><code>A</code></td><td>broadcasting 함정</td><td>loss 직전 shape 같게</td><td><code>y.unsqueeze(1)</code></td></tr>
<tr><td><code>A</code></td><td>행렬 곱</td><td><code>[batch, in] @ [in, out]</code></td><td><code>nn.Linear</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>MSE</td><td>차이 → 제곱 → 평균</td><td><code>nn.MSELoss()</code></td></tr>
<tr><td><code>A</code></td><td>autograd</td><td>backward는 기울기만 · 누적 → reset</td><td><code>optimizer.zero_grad()</code></td></tr>
<tr><td><code>A</code></td><td>업데이트 · 학습률</td><td><code>no_grad</code> 안 빼기 · 경로로 판단</td><td><code>optimizer.step()</code></td></tr>
<tr><td><code>B</code></td><td>NumPy 연결</td><td>from_numpy는 공유</td><td><code>torch.from_numpy(a)</code></td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 학습은 무엇을 숫자로 만드는 데서 시작하는가?
- **1-3** · shape가 같으면 같은 계산에 넣어도 되는가?
- **1-4** · 범주 번호에 무조건 `.float()`를 붙이면?
- **2-2** · `weather[1]`과 `weather[1:2]`의 shape는?
- **3-2** · `reshape(4, 3)`은 transpose인가?
- **3-4** · `cat`과 `stack`은 어떻게 고르는가?
- **4-1** · `[n, 1] - [n]`은 왜 위험한가?
- **4-2** · scalar loss가 나오면 짝이 맞은 것인가?
- **4-4** · `(7, 4) @ (3, 2)`가 실패하는 이유는?
- **5-3** · MSE가 같으면 오차 패턴도 같은가?
- **6-2** · `backward()`가 하는 일과 하지 않는 일은?
- **6-3** · 같은 loss로 backward 두 번은?
- **6-5** · `.grad`를 비우지 않으면?
- **7-1** · parameter가 바뀌는 순간은?
- **7-4** · 학습률을 마지막 loss 하나로 고르면 안 되는 이유는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `torch.arange(8)`을 샘플 2 × 특성 4로 바꾼 뒤 두 번째 샘플을 `[batch, feature]` 모양으로 꺼내 shape 출력
2. 예측 `[[1.], [2.]]` · 정답 `[1., 4.]`의 MSE를 1:1 짝으로 계산
3. w = 0 · b = 0 · x = [1, 2] · 정답 [3, 5]에서 학습률 0.1로 한 단계 뒤 w 출력

In [32]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 4-1의 정답을 `unsqueeze(1)` 대신 예측 쪽을 `squeeze(1)`해 1:1로 만들면 loss는?
2. 7-4의 실험 함수에 학습률 0.25를 넣으면 12단계 경로는 감소 · 증가 중 어느 쪽인가?
3. 8장 새 데이터에서 학습률 0.1로 3단계 하면 0.05보다 loss가 더 작을까?

In [33]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 예측과 실제값의 차이
- **1-3** · 아님 · 축의 의미까지 확인
- **1-4** · 범주라는 역할이 숨겨짐
- **2-2** · `(3,)` · `(1, 3)`
- **3-2** · 아님 · 순서대로 다시 채움
- **3-4** · 기존 축을 늘리면 cat · 새 축이 필요하면 stack
- **4-1** · 에러 없이 `(n, n)` 조합 비교
- **4-2** · 아님 · 차이 텐서 shape로 확인
- **4-4** · 안쪽 축 4 ≠ 3
- **5-3** · 아님 · 평균이 감춤
- **6-2** · 기울기 계산 → `.grad` · 값은 안 바꿈
- **6-3** · RuntimeError · 매번 fresh forward
- **6-5** · 기울기 누적 (-13 → -26)
- **7-1** · `no_grad` 안의 `w -= lr * w.grad`
- **7-4** · 경로가 느림 · 감소 · 발산인지 봐야 함

#### 코드 문제 답

```python
t = torch.arange(8).reshape(2, 4)
print(tuple(t[1:2].shape))  # (1, 4)

p = torch.tensor([[1.0], [2.0]])
y = torch.tensor([1.0, 4.0])
print(((p - y.unsqueeze(1)) ** 2).mean().item())  # 2.0

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = ((w * torch.tensor([1.0, 2.0]) + b - torch.tensor([3.0, 5.0])) ** 2).mean()
loss.backward()
with torch.no_grad():
    w -= 0.1 * w.grad
print(round(w.item(), 4))  # 1.3
```

#### 변형 문제 답

*문제 → 답*

- **1** · `(prediction_column.squeeze(1) - target_flat)` → `(4,)` · loss 1.0 (같은 1:1 짝)
- **2** · 매번 감소 · 17 → 8.6562 → … → 0.0114 · 0.2(0.001533)보다 덜 줄어듦 → 크다고 빠른 것이 아님
- **3** · 더 작음 · 0.1495 (0.05는 0.2098) · 경로 0.4 → 0.1664 → 0.1495로 줄어드는 폭은 점점 작아짐

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) <code>backward()</code> 뒤 w가 바뀐다고 답함</td><td>기울기 계산과 업데이트를 섞음</td><td>값은 <code>no_grad</code> 안의 <code>-=</code>에서 바뀜</td><td>이 줄은 기울기만 구하나, 값을 바꾸나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. loss가 scalar로 잘 나왔는데도 계산이 틀릴 수 있나요?**
> - 있습니다.
> - 예측이 [n, 1], 정답이 [n]이면 broadcasting으로 [n, n] 조합이 만들어지고, mean이 그 결과를 숫자 하나로 줄여 에러 없이 loss가 나옵니다.
> - 그래서 loss 직전에 예측과 정답 shape, 그리고 차이 텐서의 shape를 확인합니다.

**Q2. reshape와 transpose의 차이는?**
> - reshape는 원소 수만 같으면 값을 순서대로 다시 채워 새 모양을 만들고, 축의 의미는 보장하지 않습니다.
> - transpose는 축의 순서를 바꿔 행과 열의 의미를 맞바꿉니다.
> - 같은 shape가 나와도 값 배치가 다르므로 의미를 바꿀 때는 transpose를 씁니다.

**Q3. backward()는 무엇을 하나요?**
> - forward 때 기록된 계산을 거꾸로 따라가 각 parameter의 기울기를 .grad에 저장합니다.
> - parameter 값은 바꾸지 않고, 업데이트는 no_grad 안에서 따로 합니다.
> - .grad는 누적되므로 다음 단계 전에 비워야 합니다.

**Q4. 학습률은 어떻게 판단하나요?**
> - 기울기가 방향을, 학습률이 한 번의 이동 크기를 정합니다.
> - 너무 작으면 진전이 느리고 너무 크면 최저점을 지나쳐 loss가 커집니다.
> - 같은 데이터와 시작값에서 학습률만 바꿔 loss 경로 전체를 비교하고, 마지막 값 하나로 결론 내리지 않습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>torch.tensor()</code> · <code>zeros()</code> · <code>ones()</code> · <code>arange()</code></td><td>torch</td><td>값 · 틀 · 수열</td><td>1-2 <code>torch.arange(0, 6, 2)</code></td></tr>
<tr><td><code>rand()</code> · <code>randn()</code> · <code>manual_seed()</code></td><td>torch</td><td>난수 · seed</td><td>1-2 <code>torch.manual_seed(7)</code></td></tr>
<tr><td><code>.shape</code> · <code>.ndim</code> · <code>.dtype</code> · <code>.device</code></td><td>텐서 속성</td><td>모양 · 축 수 · 종류 · 장치</td><td>1-3 <code>value.ndim</code> · 1-4 <code>temperatures.dtype</code></td></tr>
<tr><td><code>t[i]</code> · <code>t[:, j]</code> · <code>t[a:b]</code> · <code>t[mask]</code></td><td>인덱싱</td><td>선택</td><td>2-1 <code>weather[weather[:, 2] == 1]</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>reshape()</code> · <code>-1</code> · <code>numel()</code></td><td>텐서 메서드</td><td>모양 바꾸기</td><td>3-1 <code>blocks.reshape(2, -1)</code></td></tr>
<tr><td><code>.T</code></td><td>텐서 속성</td><td>축 교환</td><td>3-2 <code>score_grid.T</code></td></tr>
<tr><td><code>unsqueeze()</code> · <code>squeeze()</code></td><td>텐서 메서드</td><td>길이 1 축</td><td>3-3 <code>target_flat.unsqueeze(1)</code></td></tr>
<tr><td><code>torch.cat()</code> · <code>torch.stack()</code></td><td>torch</td><td>잇기 · 쌓기</td><td>3-4 <code>torch.stack([sensor_a, sensor_b], dim=0)</code></td></tr>
<tr><td><code>from_numpy()</code> · <code>.numpy()</code></td><td>torch · 텐서 메서드</td><td>NumPy 연결</td><td>3-5 <code>torch.from_numpy(numpy_values)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>*</code> · <code>@</code> · <code>torch.matmul()</code></td><td>torch 연산</td><td>원소별 · 행렬 곱</td><td>4-3 <code>X @ W</code></td></tr>
<tr><td><code>(diff ** 2).mean()</code></td><td>torch 연산</td><td>MSE</td><td>5-2 <code>squared_error.mean()</code></td></tr>
<tr><td><code>requires_grad</code> · <code>.backward()</code> · <code>.grad</code> · <code>.grad_fn</code></td><td>autograd</td><td>기록 · 기울기</td><td>6-2 <code>loss.backward()</code></td></tr>
<tr><td><code>.grad.zero_()</code></td><td>텐서 메서드</td><td>기울기 비우기</td><td>6-5 <code>trace_w.grad.zero_()</code></td></tr>
<tr><td><code>torch.no_grad()</code> · <code>.detach()</code> · <code>.item()</code></td><td>torch · 텐서 메서드</td><td>기록 끄기 · 떼기 · 숫자</td><td>7-1 <code>with torch.no_grad():</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>w -= lr * w.grad</code></td><td>torch 연산 (제자리)</td><td>업데이트</td><td>7-1 <code>w -= step_size * w.grad</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>loss 직전 예측 · 정답 shape 비교</td><td><code>[n, 1] - [n]</code> 그대로</td></tr>
<tr><td>축 의미를 문장으로 말하기</td><td>shape 숫자만 같으면 같은 계산</td></tr>
<tr><td>의미를 바꿀 땐 <code>.T</code> · <code>transpose</code></td><td><code>reshape</code>로 행 · 열 바꾸기</td></tr>
<tr><td>매 단계 fresh forward · reset</td><td>같은 loss로 backward 두 번</td></tr>
<tr><td>학습률은 경로 전체로 판단</td><td>마지막 loss 하나로 결론</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>x[1]</code> vs <code>x[1:2]</code></td><td>축 사라짐 vs 유지</td><td>배치 모양이면 범위</td></tr>
<tr><td><code>reshape</code> vs <code>.T</code></td><td>다시 채움 vs 축 교환</td><td>의미는 <code>.T</code></td></tr>
<tr><td><code>cat</code> vs <code>stack</code></td><td>기존 축 vs 새 축</td><td>새 축 이름을 말할 수 있으면 stack</td></tr>
<tr><td><code>*</code> vs <code>@</code></td><td>같은 위치 곱 vs 곱해서 합</td><td>층은 <code>@</code></td></tr>
<tr><td><code>backward</code> vs 업데이트</td><td>기울기 계산 vs 값 변경</td><td>값은 <code>no_grad</code> 안 <code>-=</code></td></tr>
<tr><td>누적 vs <code>zero_()</code></td><td>더해짐 vs 비움</td><td>매 단계 비우기</td></tr>
<tr><td><code>no_grad</code> vs <code>detach</code> vs <code>item</code></td><td>블록 vs 텐서 하나 vs 숫자</td><td>업데이트는 <code>no_grad</code></td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>15분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 4장 그림을 가리고 <code>[4, 1] - [4]</code> 결과 shape 말하기</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>7-3 학습 loop를 빈 셀에 직접 다시 짜고 4단계 w를 먼저 예측</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 3(02_pytorch) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `01_pytorch.ipynb` 165셀을 원본 순서 그대로 7장 24섹션으로
- 학생 빈칸 셀 11개(`None` · `""` · `False`)는 해당 섹션 코드 안에 답을 채워 실행
- 원본 §9의 '새 숫자 적용' · '전체 흐름 코드 읽기'는 8장 종합 실습으로
- 원본의 그림 셀(assets/*.svg)은 원본 폴더에 파일이 없어 같은 뜻의 그림을 새로 그림
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>Prelude · 셀 0 ~ 4</td><td>예측 6 · 실제 10 · 차이</td><td>1-1</td></tr>
<tr><td>§1 · 셀 5 ~ 22</td><td>Tensor 만들기 · shape · 축 의미 · dtype · device · 실습(셀 19)</td><td>1-2 ~ 1-4</td></tr>
<tr><td>§2 · 셀 23 ~ 33</td><td>행 · 열 · 구간 · 조건 · <code>[1]</code> vs <code>[1:2]</code> · 실습(셀 32 습도)</td><td>2-1 · 2-2</td></tr>
<tr><td>§3 · 셀 34 ~ 49</td><td>reshape · <code>.T</code> · squeeze · cat/stack · 실습(셀 47) · NumPy</td><td>3-1 ~ 3-5</td></tr>
<tr><td>§4 · 셀 50 ~ 63</td><td>broadcasting 침묵 실패 · loss 직전 shape · 실습(셀 63)</td><td>4-1 · 4-2</td></tr>
<tr><td>§4 ~ 5 · 셀 64 ~ 82</td><td><code>*</code> vs <code>@</code> · 행렬 곱 계약 · 두 문제 찾기(셀 80 빈칸)</td><td>4-3 · 4-4</td></tr>
<tr><td>§6 ~ 7 · 셀 83 ~ 98</td><td><code>ŷ = wx + b</code> · MSE · 같은 MSE 다른 패턴</td><td>5-1 ~ 5-3</td></tr>
<tr><td>§8 · 셀 99 ~ 122</td><td>국소 기울기 · Autograd · 두 번 backward · 부호 · 누적 · reset</td><td>6-1 ~ 6-5</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>§9 · 셀 123 ~ 144</td><td>수동 update(셀 125 빈칸) · no_grad/detach/item · loop · 망가진 loop(셀 142) · 순서(셀 144)</td><td>7-1 ~ 7-3</td></tr>
<tr><td>§10 · 셀 145 ~ 154</td><td>학습률 0.001 · 0.1 · 0.5 · 0.2 · 0.3 · 내 학습률(셀 154)</td><td>7-4 · 변형 문제 2</td></tr>
<tr><td>§9 · 셀 136 ~ 138 · 155 ~ 157</td><td>새 숫자 적용(빈칸) · 코드 읽기(셀 156 빈칸)</td><td>8</td></tr>
<tr><td>셀 158 ~ 164</td><td>틀린 문장 고치기 · Closing 자기 말 설명</td><td>기억 확인 문제 · 면접</td></tr>
</tbody></table>

### 전체 코드

1 ~ 8장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 예측과 실제값의 차이
2. 1-2 텐서 만들기
3. 1-3 축과 shape

In [34]:
# ---------- 1-1 ----------
# 원본 Prelude(셀 3): 입력 2 · 현재 규칙 3 · 실제값 10
# 왜 · 학습이 '예측과 실제값의 차이를 줄이는 일'이라는 것을 숫자 하나로 먼저 보기 위해
input_value = 2.0     # 입력
current_weight = 3.0  # 지금의 내부 값 (규칙)
actual_value = 10.0   # 실제값 (정답)

# 왜 · 예측과 실제값의 차이를 직접 계산해 손실의 출발점을 보기 위해
prediction = current_weight * input_value  # 1. 예측 만들기
difference = prediction - actual_value     # 2. 실제값과 얼마나 다른지

print("입력:", input_value)      # 2.0
print("현재 예측:", prediction)  # 3 × 2
print("실제값:", actual_value)   # 10.0
print("차이:", difference)       # 음수 = 예측이 작음


# 3 · 4단계 미리보기: w를 조금 키우면 차이가 줄어드나?
# 왜 · 규칙(w)을 조금 바꾸면 차이가 줄어드는지 미리 확인해 경사하강의 감을 잡기 위해
for w_try in [3.0, 4.0, 5.0]:  # 내부 값 후보
    # 새 예측
    print(f"w={w_try} → 예측 {w_try * input_value} · 차이 {w_try * input_value - actual_value}")


# ---------- 1-2 ----------
# 원본 셀 8: 같은 seed로 여섯 가지 만들기
# 왜 · seed를 고정한 뒤 여러 방식으로 텐서를 만들어 결과를 다시 얻을 수 있게 하기 위해
torch.manual_seed(7)  # 난수를 다시 확인할 수 있게 고정

# 왜 · 값을 직접 지정한 텐서와 자동 생성 텐서를 비교하기 위해
known_values = torch.tensor([2.0, 4.0, 6.0])  # 값을 이미 알 때
zero_frame = torch.zeros(2, 3)                # 구조만 · 0으로
one_frame = torch.ones(2, 3)                  # 구조만 · 1로
ordered_values = torch.arange(0, 6, 2)        # 0부터 2씩 (6 제외)
uniform_samples = torch.rand(2, 3)            # 0 이상 1 미만 균등분포
normal_samples = torch.randn(2, 3)            # 평균 0 · 표준편차 1


# 값은 외우지 않고 shape · dtype만 비교
# 왜 · 난수 값보다 모양 · 자료형이 코드 오류를 가르는 기준이기 때문에
created = {"직접 지정": known_values, "0으로 준비": zero_frame,           # 이름 → 텐서
           "1로 준비": one_frame, "일정 간격": ordered_values,
           "균등 난수": uniform_samples, "정규 난수": normal_samples}
for label, value in created.items():                                      # 여섯 텐서를 차례로
    print(f"{label:8s} shape={tuple(value.shape)} dtype={value.dtype}")   # 모양 · 종류
print("arange 값:", ordered_values.tolist())                              # stop 6은 빠짐
print("rand 첫 행:", [round(v, 4) for v in uniform_samples[0].tolist()])  # seed 7 → 같은 환경에서 매번 같은 값


# ---------- 1-3 ----------
# 원본 셀 11: 0차원 ~ 4차원
# 왜 · 축 개수가 늘어날 때 shape가 어떻게 바뀌는지 한 번에 비교하기 위해
scalar = torch.tensor(7.0)                                               # 숫자 하나
vector = torch.tensor([18.0, 21.0, 25.0])                                # 축 1개
matrix = torch.tensor([[18.0, 45.0, 0.0],                                # 관측 2 × 특성 3
                       [21.0, 55.0, 1.0]])
mini_images = torch.arange(24, dtype=torch.float32).reshape(2, 3, 2, 2)  # (N, C, H, W)

for name, value in {"scalar": scalar, "vector": vector,                 # 네 텐서를 차례로
                    "matrix": matrix, "mini_images": mini_images}.items():
    print(f"{name:12s} shape={tuple(value.shape)}, ndim={value.ndim}")  # 축 길이 · 축 개수


# 원본 셀 15: 같은 shape · 다른 의미
# 왜 · 같은 shape라도 축의 의미가 다를 수 있다는 것을 확인하기 위해
student_subject_scores = torch.tensor([[80.0, 90.0, 70.0], [75.0, 85.0, 95.0]])  # 학생 2 × 과목 3
sample_class_scores = torch.tensor([[2.1, 0.3, -0.4], [0.2, 1.8, 0.1]])  # 샘플 2 × class 점수 3
print("shape만 같은가?:", student_subject_scores.shape == sample_class_scores.shape)  # 숫자만 비교

입력: 2.0
현재 예측: 6.0
실제값: 10.0
차이: -4.0
w=3.0 → 예측 6.0 · 차이 -4.0
w=4.0 → 예측 8.0 · 차이 -2.0
w=5.0 → 예측 10.0 · 차이 0.0
직접 지정    shape=(3,) dtype=torch.float32
0으로 준비   shape=(2, 3) dtype=torch.float32
1로 준비    shape=(2, 3) dtype=torch.float32
일정 간격    shape=(3,) dtype=torch.int64
균등 난수    shape=(2, 3) dtype=torch.float32
정규 난수    shape=(2, 3) dtype=torch.float32
arange 값: [0, 2, 4]
rand 첫 행: [0.5349, 0.1988, 0.6592]
scalar       shape=(), ndim=0
vector       shape=(3,), ndim=1
matrix       shape=(2, 3), ndim=2
mini_images  shape=(2, 3, 2, 2), ndim=4
shape만 같은가?: True


**실행 흐름**

1. 1-4 이미지 축 읽기 · 자료형 차이

In [35]:
# 원본 셀 19 실습 답: mini_images 축을 문장으로
# 왜 · 4차원 이미지 텐서의 축을 말로 풀어 읽는 연습을 하기 위해
n, c, h, w = mini_images.shape                                 # 축 네 개를 이름으로 받기
print(f"샘플 {n}개, 샘플마다 채널 {c}개, 높이 {h}, 너비 {w}")  # 실습 빈칸 답


# ---------- 1-4 ----------
# 원본 셀 17: 정수만 넣은 값 vs 실수
# 왜 · 정수만 넣으면 정수 텐서가 되어 계산 결과가 달라질 수 있음을 보기 위해
temperatures = torch.tensor([18, 21, 25])                                # 정수만
measurements = torch.tensor([[18.0, 45.0], [21.0, 55.0], [25.0, 60.0]])  # 실수
# 종류 · 위치 · 모양
print("temperatures:", temperatures.dtype, temperatures.device, tuple(temperatures.shape))
# 같은 세 가지
print("measurements:", measurements.dtype, measurements.device, tuple(measurements.shape))


# 원본 셀 21: 같은 0 · 1 · 2라도 역할이 다름
# 왜 · 같은 숫자라도 측정값 · 번호 · 정답처럼 역할에 따라 자료형이 달라야 함을 보이기 위해
measurement_values = torch.tensor([0.0, 1.0, 2.0])                    # 연속 측정값 → 실수
category_ids = torch.tensor([0, 1, 2])                                # 범주 번호 → 정수
for name, value in {"measurement_values": measurement_values,         # 두 텐서를 차례로
                    "category_ids": category_ids}.items():
    print(f"{name:18s} values={value.tolist()} dtype={value.dtype}")  # 값 · 종류


# 원본 셀 22 질문 3: 무조건 .float()를 붙이면 숨겨지는 것
# 왜 · 무조건 실수로 바꾸면 정답 번호의 의미가 숨겨지는 문제를 확인하기 위해
print("float 뒤:", category_ids.float().dtype, category_ids.float().tolist())  # 번호가 실수가 됨

샘플 2개, 샘플마다 채널 3개, 높이 2, 너비 2
temperatures: torch.int64 cpu (3,)
measurements: torch.float32 cpu (3, 2)
measurement_values values=[0.0, 1.0, 2.0] dtype=torch.float32
category_ids       values=[0, 1, 2] dtype=torch.int64
float 뒤: torch.float32 [0.0, 1.0, 2.0]


**실행 흐름**

1. 2-1 관측 × 특성 인덱싱
2. 2-2 같은 값 · 다른 shape

In [36]:
# ---------- 2-1 ----------
# 원본 셀 24: 관측 4 × 특성 3 (온도 · 습도 · 비 여부)
# 왜 · 관측 × 특성 표에서 행 · 열의 의미를 정해 두기 위해
weather = torch.tensor([[18.0, 45.0, 0.0],   # 관측 1
                        [21.0, 55.0, 1.0],   # 관측 2
                        [25.0, 60.0, 1.0],   # 관측 3
                        [20.0, 50.0, 0.0]])  # 관측 4
feature_names = ["온도", "습도", "비 여부"]  # axis 1의 이름
print("shape:", tuple(weather.shape))        # (4, 3)


# 원본 셀 25: 행 · 열 · 구간 · 조건
# 왜 · 행 · 열 · 구간 · 조건으로 필요한 값을 꺼내는 방법을 비교하기 위해
print("두 번째 관측 전체:", weather[1].tolist())               # axis 0에서 하나
print("모든 관측의 온도:", weather[:, 0].tolist())             # 모든 행 · 열 0
print("앞의 두 관측, 온도와 습도:", weather[:2, :2].tolist())  # 두 축 범위
print("비가 온 관측:", weather[weather[:, 2] == 1].tolist())   # 조건 True인 행만


# 원본 셀 31 ~ 33 실습: 모든 관측의 습도만
# 왜 · 특성 하나만 모든 관측에서 꺼내는 대표 상황을 연습하기 위해
humidity_answer = weather[:, feature_names.index("습도")]                         # 열 위치 1
print("습도:", humidity_answer.tolist(), "shape:", tuple(humidity_answer.shape))  # (4,)


# ---------- 2-2 ----------
# 원본 셀 28: 같은 값 · 다른 shape
# 왜 · 같은 값이라도 축 수가 다르면 모델이 다르게 받아들인다는 것을 보이기 위해
one_observation = weather[1]                                      # 정수 하나 → 관측 축 사라짐
one_observation_batch = weather[1:2]                              # 범위 → 길이 1 관측 축 유지
print("weather[1] 값:", one_observation.tolist())                 # 세 값
print("weather[1] shape:", tuple(one_observation.shape))          # (3,)
print("weather[1:2] 값:", one_observation_batch.tolist())         # 같은 세 값을 한 겹 더
print("weather[1:2] shape:", tuple(one_observation_batch.shape))  # (1, 3)


# 원본 셀 30 빈칸 답: [feature]를 기대하면 [1], [batch, feature]면 [1:2]
# 왜 · 모델이 기대하는 모양에 따라 인덱싱 방법을 골라야 하기 때문에
# 1 vs 2
print("특성 벡터:", one_observation.ndim, "차원 | 배치 1개:", one_observation_batch.ndim, "차원")
# 축 추가 = 범위 선택
print("unsqueeze(0)로도 같음:", torch.equal(one_observation.unsqueeze(0), one_observation_batch))

shape: (4, 3)
두 번째 관측 전체: [21.0, 55.0, 1.0]
모든 관측의 온도: [18.0, 21.0, 25.0, 20.0]
앞의 두 관측, 온도와 습도: [[18.0, 45.0], [21.0, 55.0]]
비가 온 관측: [[21.0, 55.0, 1.0], [25.0, 60.0, 1.0]]
습도: [45.0, 55.0, 60.0, 50.0] shape: (4,)
weather[1] 값: [21.0, 55.0, 1.0]
weather[1] shape: (3,)
weather[1:2] 값: [[21.0, 55.0, 1.0]]
weather[1:2] shape: (1, 3)
특성 벡터: 1 차원 | 배치 1개: 2 차원
unsqueeze(0)로도 같음: True


**실행 흐름**

1. 3-1 reshape
2. 3-2 reshape vs 전치
3. 3-3 길이 1 축 넣고 빼기

In [37]:
import numpy as np  # NumPy 배열


# ---------- 3-1 ----------
# 원본 셀 35: 24개 값의 모양만 바꾸기
# 왜 · 값은 그대로 두고 모양만 바꾸는 여러 방법을 비교하기 위해
sequence = torch.arange(24)                   # 0 ~ 23
blocks = sequence.reshape(2, 3, 4)            # 2 × 3 × 4 = 24
flattened_per_sample = blocks.reshape(2, -1)  # 샘플은 두고 나머지 펴기
b = blocks.reshape(3, -1)                     # 24 ÷ 3 = 8

for name, value in [("원본", sequence), ("blocks", blocks),                  # 네 텐서를 차례로
                    ("샘플별 펼침", flattened_per_sample), ("reshape(3, -1)", b)]:
    print(f"{name:14s} shape={tuple(value.shape)} 원소 수={value.numel()}")  # 원소 수는 그대로


# 원소 수가 맞지 않으면 실패
# 왜 · reshape가 실패하는 조건을 에러로 확인하기 위해
try:
    sequence.reshape(5, 5)                                               # 25칸 ≠ 24개
except RuntimeError as error:                                            # 에러를 잡아 멈추지 않게
    print("reshape(5, 5) → RuntimeError:", str(error).split(" is ")[0])  # 메시지 앞부분


# ---------- 3-2 ----------
# 원본 셀 37: 학생 3 × 문항 4
# 왜 · reshape가 축을 바꾸지 않고 값을 순서대로 다시 채운다는 것을 확인하기 위해
score_grid = torch.arange(1, 13).reshape(3, 4)         # 1 ~ 12
possible_but_reinterpreted = score_grid.reshape(4, 3)  # 순서대로 다시 채움
axes_swapped = score_grid.T                            # 행 ↔ 열 교환

# 왜 · reshape와 전치의 결과가 다르다는 것을 첫 행으로 비교하기 위해
print("원본 첫 행:", score_grid[0].tolist())         # 학생 1의 문항 4개
print("reshape(4, 3) 첫 행:", possible_but_reinterpreted[0].tolist())  # 앞에서부터 3개
print("transpose 첫 행:", axes_swapped[0].tolist())  # 원래 첫 열 = 문항 1의 학생 3명
# 같은 shape · 다른 값
print("두 결과의 값 배치가 같은가?:", torch.equal(possible_but_reinterpreted, axes_swapped))


# 원본 셀 38: 세 기준(실행 가능 · 의미 보존 · 축 교환)으로 판정
# 왜 · 실행 가능 여부와 의미 보존을 따로 판정해야 한다는 것을 확인하기 위해
for label, shape in [("reshape(3, 4)", (3, 4)), ("reshape(2, 6)", (2, 6))]:  # 두 후보
    ok = score_grid.numel() == shape[0] * shape[1]                           # 원소 수 계약
    print(f"{label}: 실행 가능 {ok}")                                        # 둘 다 12칸


# ---------- 3-3 ----------
# 원본 셀 39: 길이 1 축 넣고 빼기
# 왜 · 길이 1 축을 넣고 빼서 모델이 원하는 모양에 맞추기 위해
target_flat = torch.tensor([3.0, 5.0, 7.0, 9.0])    # 정답 (4,)
target_column = target_flat.unsqueeze(1)            # 끝에 길이 1 축
target_restored = target_column.squeeze(1)          # 그 축을 다시 제거
print("flat:", tuple(target_flat.shape))            # (4,)
print("unsqueeze(1):", tuple(target_column.shape))  # (4, 1)
print("squeeze(1):", tuple(target_restored.shape))  # (4,)


# 원본 셀 46 ~ 47 실습 답: 12개 → 샘플 3 × 특성 4
# 왜 · 값 12개를 샘플 × 특성 모양으로 바꾸는 연습을 하기 위해
practice_values = torch.arange(12)                # 0 ~ 11
practice_matrix = practice_values.reshape(3, -1)  # 둘째 축은 자동 4
print("shape:", tuple(practice_matrix.shape), "원소 수:", practice_matrix.numel())  # 12 유지

원본             shape=(24,) 원소 수=24
blocks         shape=(2, 3, 4) 원소 수=24
샘플별 펼침         shape=(2, 12) 원소 수=24
reshape(3, -1) shape=(3, 8) 원소 수=24
reshape(5, 5) → RuntimeError: shape '[5, 5]'
원본 첫 행: [1, 2, 3, 4]
reshape(4, 3) 첫 행: [1, 2, 3]
transpose 첫 행: [1, 5, 9]
두 결과의 값 배치가 같은가?: False
reshape(3, 4): 실행 가능 True
reshape(2, 6): 실행 가능 True
flat: (4,)
unsqueeze(1): (4, 1)
squeeze(1): (4,)
shape: (3, 4) 원소 수: 12


**실행 흐름**

1. 3-4 쌓기 vs 잇기
2. 3-5 NumPy ↔ 텐서

In [38]:
# ---------- 3-4 ----------
# 원본 셀 41: 오전 · 오후 기록 (시간 2 × 특성 2)
# 왜 · 같은 모양의 텐서를 새 축으로 쌓을지 기존 축으로 이을지 비교하기 위해
morning = torch.tensor([[18.0, 45.0], [21.0, 55.0]])                    # 오전 두 관측
afternoon = torch.tensor([[25.0, 60.0], [20.0, 50.0]])                  # 오후 두 관측
print("cat:", tuple(torch.cat([morning, afternoon], dim=0).shape))      # 기존 축 늘리기
print("stack:", tuple(torch.stack([morning, afternoon], dim=0).shape))  # 새 축 만들기


# 원본 셀 44: 두 센서 기록
# 왜 · 두 센서 기록을 합칠 때 축 선택에 따라 의미가 달라지는 것을 보기 위해
sensor_a = torch.tensor([[10.0, 0.1], [11.0, 0.2]])                  # 센서 A
sensor_b = torch.tensor([[20.0, 0.3], [21.0, 0.4]])                  # 센서 B
long_timeline = torch.cat([sensor_a, sensor_b], dim=0)               # 하나의 긴 시간 흐름
separate_sensors = torch.stack([sensor_a, sensor_b], dim=0)          # 센서 축을 새로
# 시간 4 × 특성 2
print("기존 시간 축을 늘린 결과:", tuple(long_timeline.shape), "ndim", long_timeline.ndim)
# 센서 2 × 시간 2 × 특성 2
print("새 센서 축을 만든 결과:", tuple(separate_sensors.shape), "ndim", separate_sensors.ndim)
print("센서 B만:", separate_sensors[1, :, 0].tolist(), "(첫 특성)")  # 새 축 번호 1 = 센서 B


# ---------- 3-5 ----------
# 원본 셀 49: NumPy ↔ 텐서
# 왜 · NumPy 배열과 텐서를 오가는 방법을 확인하기 위해

# 왜 · 자료형이 그대로 이어지는지 확인하기 위해
numpy_values = np.array([1.5, 2.5, 3.5], dtype=np.float32)  # 모델용 float32로 만든 배열
torch_values = torch.from_numpy(numpy_values)               # 배열 → 텐서 (메모리 공유)
back_to_numpy = torch_values.numpy()                        # 텐서 → 배열
print(type(numpy_values).__name__, numpy_values.dtype)      # ndarray float32
print(type(torch_values).__name__, torch_values.dtype)      # Tensor torch.float32
print(type(back_to_numpy).__name__, back_to_numpy.dtype)    # 다시 ndarray


# 보강: from_numpy는 메모리를 공유 · torch.tensor는 복사
# 왜 · 한쪽을 고치면 다른 쪽도 바뀌는 공유 함정을 미리 보기 위해
copied = torch.tensor(numpy_values)          # 수정 전에 복사해 둔 텐서
numpy_values[0] = 100.0                      # 원본 배열만 수정
print("from_numpy:", torch_values.tolist())  # 공유 → 100이 보임
print("tensor 복사:", copied.tolist())       # 복사본은 그대로

cat: (4, 2)
stack: (2, 2, 2)
기존 시간 축을 늘린 결과: (4, 2) ndim 2
새 센서 축을 만든 결과: (2, 2, 2) ndim 3
센서 B만: [20.0, 21.0] (첫 특성)
ndarray float32
Tensor torch.float32
ndarray float32
from_numpy: [100.0, 2.5, 3.5]
tensor 복사: [1.5, 2.5, 3.5]


**실행 흐름**

1. 4-1 예측 · 정답 모양 맞추기
2. 4-2 브로드캐스팅
3. 4-3 모양 검사

In [39]:
# ---------- 4-1 ----------
# 원본 셀 51: 예측 [n, 1] · 정답 [n]
# 왜 · 예측 [n, 1]과 정답 [n]의 모양 차이가 조용한 버그를 만드는 상황을 재현하기 위해
prediction_column = torch.tensor([[2.0], [4.0], [6.0], [8.0]])  # (4, 1)
target_flat = torch.tensor([3.0, 5.0, 7.0, 9.0])                # (4,)
silent_difference = prediction_column - target_flat             # 에러 없이 계산됨
print("difference shape:", tuple(silent_difference.shape))      # (4, 4) 모든 조합
print("첫 행:", silent_difference[0].tolist())                  # 예측 2가 정답 4개와 비교됨


# 원본 셀 53: 정답에 축 추가 → 1:1
# 왜 · 정답에 축을 하나 더해 예측과 1:1로 맞추기 위해
target_column = target_flat.unsqueeze(1)             # (4, 1)
safe_difference = prediction_column - target_column  # 같은 위치끼리
# 모두 -1
print("1:1 difference shape:", tuple(safe_difference.shape), safe_difference.squeeze(1).tolist())


# 원본 셀 55 · 56 · 57: 의도한 broadcasting
# 왜 · 의도한 브로드캐스팅과 의도하지 않은 브로드캐스팅을 구분하기 위해
m1 = torch.tensor([[3.0, 3.0]])                                 # (1, 2)
m2 = torch.tensor([[2.0, 2.0]])                                 # (1, 2)
print("(1,2)+(1,2):", (m1 + m2).tolist())                       # 같은 모양
m3 = torch.tensor([[1.0, 2.0]]) + torch.tensor([[3.0], [4.0]])  # (1, 2) + (2, 1)
print("(1,2)+(2,1):", tuple(m3.shape), m3.tolist())             # 둘 다 늘어나 (2, 2)
# 모든 값에 0.5
print("행렬 + scalar:", (torch.tensor([[1.0, 2.0], [3.0, 4.0]]) + torch.tensor(0.5)).tolist())


# ---------- 4-2 ----------
# 원본 셀 59: 잘못된 짝과 올바른 짝 모두 scalar loss
# 왜 · 모양이 틀려도 손실이 숫자 하나로 나와 에러 없이 틀린다는 것을 보이기 위해
wrong_difference = prediction_column - target_flat               # (4, 4)
right_difference = prediction_column - target_flat.unsqueeze(1)  # (4, 1)
wrong_scalar_loss = (wrong_difference ** 2).mean()               # 16개 평균
right_scalar_loss = (right_difference ** 2).mean()               # 4개 평균
print("잘못된 경로: difference", tuple(wrong_difference.shape),  # 짝 확인
      "→ loss", tuple(wrong_scalar_loss.shape), wrong_scalar_loss.item())
print("1:1 경로: difference", tuple(right_difference.shape),     # 짝 확인
      "→ loss", tuple(right_scalar_loss.shape), right_scalar_loss.item())


# 원본 셀 63 실습 답: shape guard 통과
# 왜 · 손실 계산 전에 모양을 assert로 검사해 조용한 버그를 막기 위해
target_attempt = target_flat.unsqueeze(1)            # 정답에 길이 1 축
if target_attempt.shape == prediction_column.shape:  # 계산 전에 비교
    print("안전한 1:1 비교 준비됨")                  # 통과 문장
else:
    # 실패 문장
    print("shape를 다시 확인함:", tuple(prediction_column.shape), tuple(target_attempt.shape))


# ---------- 4-3 ----------
# 원본 셀 64: 같은 위치 곱 vs 곱해서 합
# 왜 · 원소별 곱과 내적의 차이를 결과 모양으로 구분하기 위해
features = torch.tensor([2.0, 3.0])                                         # 특성 2개
weights = torch.tensor([4.0, 5.0])                                          # 가중치 2개
elementwise = features * weights                                            # [2×4, 3×5]
combined = features @ weights                                               # 2×4 + 3×5
print("* 결과:", elementwise.tolist(), "shape:", tuple(elementwise.shape))  # 값 2개
print("@ 결과:", combined.item(), "shape:", tuple(combined.shape))          # 값 1개 (0차원)

difference shape: (4, 4)
첫 행: [-1.0, -3.0, -5.0, -7.0]
1:1 difference shape: (4, 1) [-1.0, -1.0, -1.0, -1.0]
(1,2)+(1,2): [[5.0, 5.0]]
(1,2)+(2,1): (2, 2) [[4.0, 5.0], [5.0, 6.0]]
행렬 + scalar: [[1.5, 2.5], [3.5, 4.5]]
잘못된 경로: difference (4, 4) → loss () 11.0
1:1 경로: difference (4, 1) → loss () 1.0
안전한 1:1 비교 준비됨
* 결과: [8.0, 15.0] shape: (2,)
@ 결과: 23.0 shape: ()


**실행 흐름**

1. 4-4 행렬 곱과 결과 모양

In [40]:
# 원본 셀 67: [batch=3, in=2] @ [in=2, out=1]
# 왜 · 배치 입력과 가중치의 행렬 곱이 어떤 모양을 만드는지 확인하기 위해
X = torch.tensor([[1.0, 2.0], [2.0, 1.0], [3.0, 2.0]])                        # 샘플 3 × 특성 2
W = torch.tensor([[2.0], [1.0]])                                              # 특성 2 → 출력 1
batch_output = X @ W                                                          # (3, 1)
print("X @ W:", tuple(batch_output.shape), batch_output.squeeze(1).tolist())  # 손계산 4 · 5 · 8
print("matmul과 동일:", torch.equal(batch_output, torch.matmul(X, W)))        # 같은 연산


# 원본 셀 69: 특성별 기여를 더하면 X @ W
# 왜 · 행렬 곱이 특성별 기여의 합이라는 것을 직접 계산으로 확인하기 위해
contributions = X.unsqueeze(2) * W.unsqueeze(0)                           # (3, 2, 1) 기여
print("샘플별 특성 기여:", contributions.squeeze(2).tolist())             # [1×2, 2×1] ...
print("기여를 더한 결과:", contributions.sum(dim=1).squeeze(1).tolist())  # X @ W와 같음


# ---------- 4-4 ----------
# 원본 셀 73 · 75: 결과 shape를 먼저 말하기
# 왜 · 코드를 실행하기 전에 결과 모양을 먼저 말하는 습관을 들이기 위해
# 세 경우
matmul_cases = [(torch.ones(5, 3), torch.ones(3, 2)), (torch.ones(4, 6), torch.ones(6, 2)),
                (torch.ones(1, 5), torch.ones(5, 3)), (torch.ones(7, 4), torch.ones(3, 2))]
for left_case, right_case in matmul_cases:    # 경우마다
    try:
        output_case = left_case @ right_case  # 안쪽 축이 같아야
        # 바깥 축
        print(f"{tuple(left_case.shape)} @ {tuple(right_case.shape)} → {tuple(output_case.shape)}")
    except RuntimeError:                      # 안쪽 축이 다르면
        # 안쪽 축 4 ≠ 3
        print(f"{tuple(left_case.shape)} @ {tuple(right_case.shape)} → 실행할 수 없음")


# 원본 셀 78 · 80 실습 답: 문제 A · B를 각각 최소 수정
# 왜 · 모양 오류를 최소한으로 고치는 방법을 연습하기 위해
lab_features = torch.tensor([[1.0, 2.0], [2.0, 1.0], [3.0, 2.0]])  # 샘플 3 × 특성 2
lab_target = torch.tensor([4.0, 5.0, 8.0])                         # 정답 (3,)
weight_attempt = torch.tensor([[2.0], [1.0]])                      # A 수정: 입력 축 3 → 2
target_attempt = lab_target.unsqueeze(1)                           # B 수정: (3,) → (3, 1)
retry_prediction = lab_features @ weight_attempt                   # (3, 2) @ (2, 1)
assert retry_prediction.shape == target_attempt.shape              # 1:1 짝 확인
retry_loss = ((retry_prediction - target_attempt) ** 2).mean()     # MSE
# 정답과 같음
print("다시 실행 성공 · prediction", tuple(retry_prediction.shape), "loss:", retry_loss.item())

X @ W: (3, 1) [4.0, 5.0, 8.0]
matmul과 동일: True
샘플별 특성 기여: [[2.0, 2.0], [4.0, 1.0], [6.0, 2.0]]
기여를 더한 결과: [4.0, 5.0, 8.0]
(5, 3) @ (3, 2) → (5, 2)
(4, 6) @ (6, 2) → (4, 2)
(1, 5) @ (5, 3) → (1, 3)
(7, 4) @ (3, 2) → 실행할 수 없음
다시 실행 성공 · prediction (3, 1) loss: 0.0


**실행 흐름**

1. 5-1 ŷ = wx + b
2. 5-2 MSE
3. 5-3 MSE의 한계

In [41]:
# ---------- 5-1 ----------
# 원본 셀 84: 가장 작은 모델 ŷ = wx + b
# 왜 · 가장 작은 모델 ŷ = wx + b로 예측 계산을 직접 해 보기 위해
x_train = torch.tensor([1.0, 2.0])       # 입력
target_train = torch.tensor([3.0, 5.0])  # 정답 (규칙 2x + 1)
w = torch.tensor(0.0)                    # 아직 학습 전
b = torch.tensor(0.0)                    # 아직 학습 전
prediction = w * x_train + b             # 현재 예측
print("현재 w, b:", w.item(), b.item(), "| 현재 prediction:", prediction.tolist())  # 모두 0


# 원본 셀 86: 같은 x에서 w만 +1 · b만 +1
# 왜 · w와 b가 예측에 주는 영향을 따로 떼어 비교하기 위해
x_sensitivity = torch.tensor([1.0, 2.0, 4.0])                                     # 입력 3개
base_w, base_b = 2.0, 0.5                                                         # 기준 값
base_prediction = base_w * x_sensitivity + base_b                                 # 기준 예측
w_changed_prediction = (base_w + 1.0) * x_sensitivity + base_b                    # w만 +1
b_changed_prediction = base_w * x_sensitivity + (base_b + 1.0)                    # b만 +1
print("기준 prediction:", base_prediction.tolist())                               # 2.5 · 4.5 · 8.5
print("w만 +1 했을 때 변화:", (w_changed_prediction - base_prediction).tolist())  # x와 같음
print("b만 +1 했을 때 변화:", (b_changed_prediction - base_prediction).tolist())  # 모두 1


# ---------- 5-2 ----------
# 원본 셀 91: 차이 → 제곱 → 평균 = MSE
# 왜 · MSE를 계산 순서대로 풀어 손실의 뜻을 이해하기 위해
difference = prediction - target_train         # [0-3, 0-5]
squared_error = difference ** 2                # 음수 없애기
mse = squared_error.mean()                     # 숫자 하나
print("차이:", difference.tolist())            # [-3, -5]
print("차이의 제곱:", squared_error.tolist())  # [9, 25]
print("MSE:", mse.item())                      # (9 + 25) ÷ 2


# 원본 셀 94: 두 prediction 비교
# 왜 · 손실이 낮은 예측이 더 좋은 예측이라는 것을 두 예로 확인하기 위해
prediction_a = torch.tensor([2.0, 6.0])               # 오차 -1 · +1
prediction_b = torch.tensor([1.0, 5.0])               # 오차 -2 · 0
loss_a = ((prediction_a - target_train) ** 2).mean()  # (1 + 1) ÷ 2
loss_b = ((prediction_b - target_train) ** 2).mean()  # (4 + 0) ÷ 2
print("prediction A의 MSE:", loss_a.item())           # 1.0
print("prediction B의 MSE:", loss_b.item())           # 2.0
print("RMSE(A):", loss_a.sqrt().item())               # 제곱근은 다른 이름


# ---------- 5-3 ----------
# 원본 셀 96: 같은 MSE · 다른 오차 패턴
# 왜 · MSE가 같아도 오차의 모양은 다를 수 있다는 한계를 보기 위해
pattern_target = torch.zeros(4)                                          # 정답 0 네 개
pattern_a = torch.tensor([-1.0, 1.0, -1.0, 1.0])                         # 고르게 틀림
root_two = torch.sqrt(torch.tensor(2.0))                                 # √2
pattern_b = torch.tensor([0.0, 0.0, -root_two.item(), root_two.item()])  # 둘만 크게 틀림

for label, pred in {"A": pattern_a, "B": pattern_b}.items():            # 두 패턴
    errors = pred - pattern_target                                      # 샘플별 오차
    squared = errors ** 2                                               # 샘플별 제곱오차
    rounded = [round(v, 3) for v in squared.tolist()]                   # 보기 좋게 반올림
    print(f"패턴 {label} | squared={rounded} | MSE={squared.mean().item():.3f}")  # 평균만 같음
    print(f"패턴 {label} | 최대 오차={errors.abs().max().item():.3f}")  # 가장 크게 틀린 정도

현재 w, b: 0.0 0.0 | 현재 prediction: [0.0, 0.0]
기준 prediction: [2.5, 4.5, 8.5]
w만 +1 했을 때 변화: [1.0, 2.0, 4.0]
b만 +1 했을 때 변화: [1.0, 1.0, 1.0]
차이: [-3.0, -5.0]
차이의 제곱: [9.0, 25.0]
MSE: 17.0
prediction A의 MSE: 1.0
prediction B의 MSE: 2.0
RMSE(A): 1.0
패턴 A | squared=[1.0, 1.0, 1.0, 1.0] | MSE=1.000
패턴 A | 최대 오차=1.000
패턴 B | squared=[0.0, 0.0, 2.0, 2.0] | MSE=1.000
패턴 B | 최대 오차=1.414


**실행 흐름**

1. 6-1 변화율과 기울기
2. 6-2 · 6-3 backward
3. 6-4 그래프 재사용 에러

In [42]:
# ---------- 6-1 ----------
# 원본 셀 101: b = 0으로 고정하고 w 하나만 움직이기
# 왜 · 변수를 하나만 움직여 손실이 어떻게 변하는지 보고 기울기의 뜻을 잡기 위해
def loss_with_fixed_b(w_value):                                    # w 하나를 넣으면 손실
    local_prediction = w_value * x_train                           # b는 0으로 고정함
    return ((local_prediction - target_train) ** 2).mean().item()  # MSE 숫자


for nearby_w in [-0.1, 0.0, 0.1]:  # 현재 w = 0의 왼쪽 · 제자리 · 오른쪽
    print(f"w={nearby_w:+.1f} → loss={loss_with_fixed_b(nearby_w):.3f}")  # 오른쪽이 낮음

# 왜 · 양옆 차이로 구한 변화율이 기울기와 같은지 비교하기 위해
local_change_rate = (loss_with_fixed_b(0.1) - loss_with_fixed_b(-0.1)) / 0.2  # 양옆 차이 ÷ 거리
print(f"양옆 두 점으로 본 현재 근처 변화율: {local_change_rate:.3f}")         # 음수 → 키우는 방향


# ---------- 6-2 ----------
# 원본 셀 105: 바꿀 값 w · b만 추적
# 왜 · 바꿀 값만 requires_grad로 추적해 기울기를 받기 위해
w = torch.tensor(0.0, requires_grad=True)                          # 기록 시작
b = torch.tensor(0.0, requires_grad=True)                          # 기록 시작
prediction = w * x_train + b                                       # 예측도 기록됨
loss = ((prediction - target_train) ** 2).mean()                   # 손실
print("loss:", loss.item())                                        # 17.0
print("prediction의 grad_fn:", type(prediction.grad_fn).__name__)  # 마지막 연산 = 더하기
print("loss의 grad_fn:", type(loss.grad_fn).__name__)              # 마지막 연산 = 평균


# 원본 셀 107: backward 전후 값 비교
# 왜 · backward는 기울기만 계산하고 값은 바꾸지 않는다는 것을 확인하기 위해
w_before_backward = w.detach().clone()                                      # 비교용 사본
loss.backward()                                                             # 기울기 계산 → .grad
print("backward 전후 w 동일:", torch.equal(w_before_backward, w.detach()))  # 값은 그대로
print("w.grad:", w.grad.item(), "| b.grad:", b.grad.item())                 # 기울기만 생김
print("x_train 추적:", x_train.requires_grad)                               # 입력은 추적 안 함


# ---------- 6-3 ----------
# 원본 셀 109: 같은 loss로 backward 두 번
# 왜 · 한 번 쓴 계산 그래프로 backward를 두 번 하면 에러가 나는 이유를 보기 위해
graph_w = torch.tensor(0.0, requires_grad=True)                             # 새 w
graph_b = torch.tensor(0.0, requires_grad=True)                             # 새 b
graph_loss = ((graph_w * x_train + graph_b - target_train) ** 2).mean()     # forward 한 번
graph_loss.backward()                                                       # 첫 backward
print("첫 backward 완료, grad:", graph_w.grad.item(), graph_b.grad.item())  # -13 · -8

try:
    graph_loss.backward()      # 이미 쓴 기록을 다시 따라감
except RuntimeError as error:  # 에러를 잡아 멈추지 않게
    print("같은 loss의 두 번째 backward가 중단됨:", type(error).__name__)  # RuntimeError


# 새 forward로 fresh loss를 만들면 다시 가능
# 왜 · 매 단계 새 손실을 만들어야 하는 이유를 확인하기 위해
graph_w.grad.zero_()                                                           # 지난 기울기 비우기
graph_b.grad.zero_()                                                           # 지난 기울기 비우기
fresh_graph_loss = ((graph_w * x_train + graph_b - target_train) ** 2).mean()  # 새 기록
fresh_graph_loss.backward()                                                    # 성공
print("fresh forward 뒤 backward 완료, grad:", graph_w.grad.item(), graph_b.grad.item())  # 같은 값


# ---------- 6-4 ----------
# 원본 셀 112: w · b를 ±0.01 움직여 부호 확인
# 왜 · 값을 조금 움직여 손실이 줄어드는 방향이 기울기 부호와 반대인지 확인하기 위해
def loss_at(w_value, b_value):                                     # w · b를 넣으면 손실
    local_prediction = w_value * x_train + b_value                 # 예측
    return ((local_prediction - target_train) ** 2).mean().item()  # MSE 숫자

w=-0.1 → loss=18.325
w=+0.0 → loss=17.000
w=+0.1 → loss=15.725
양옆 두 점으로 본 현재 근처 변화율: -13.000
loss: 17.0
prediction의 grad_fn: AddBackward0
loss의 grad_fn: MeanBackward0
backward 전후 w 동일: True
w.grad: -13.0 | b.grad: -8.0
x_train 추적: False
첫 backward 완료, grad: -13.0 -8.0
같은 loss의 두 번째 backward가 중단됨: RuntimeError
fresh forward 뒤 backward 완료, grad: -13.0 -8.0


**실행 흐름**

1. 6-5 값을 움직여 부호 확인 · 기울기 누적

In [43]:
epsilon = 0.01                          # 아주 조금
print("현재 loss:", loss_at(0.0, 0.0))  # 17.0
# +가 낮음
print("w를 +0.01:", round(loss_at(epsilon, 0.0), 4), "| -0.01:", round(loss_at(-epsilon, 0.0), 4))
# +가 낮음
print("b를 +0.01:", round(loss_at(0.0, epsilon), 4), "| -0.01:", round(loss_at(0.0, -epsilon), 4))


# 줄어든 양 ÷ 0.01 ≈ |기울기|
# 왜 · 작게 움직인 변화율이 기울기 크기와 맞는지 숫자로 확인하기 위해
print("w 쪽 변화율:", round((loss_at(epsilon, 0.0) - 17.0) / epsilon, 2))  # ≈ -13
print("b 쪽 변화율:", round((loss_at(0.0, epsilon) - 17.0) / epsilon, 2))  # ≈ -8


# ---------- 6-5 ----------
# 원본 셀 118: reset을 빼먹으면 · 매번 하면
# 왜 · 기울기를 지우지 않으면 누적되는 함정을 직접 비교하기 위해
def collect_grad_trace(reset_each_time):             # reset 여부만 바꾼 실험
    trace_w = torch.tensor(0.0, requires_grad=True)  # 새 w
    trace_b = torch.tensor(0.0, requires_grad=True)  # 새 b
    records = []                                     # (반복, w.grad, b.grad)
    for repeat in range(1, 3):                       # 같은 현재 값에서 두 번
        trace_loss = ((trace_w * x_train + trace_b - target_train) ** 2).mean()  # fresh loss
        trace_loss.backward()                        # 기울기 → .grad에 더해짐
        records.append((repeat, trace_w.grad.item(), trace_b.grad.item()))  # 기록
        if reset_each_time:                          # 매번 비우기
            trace_w.grad.zero_()                     # w.grad = 0
            trace_b.grad.zero_()                     # b.grad = 0
    return records                                   # 두 번의 기록


print("reset 누락:", collect_grad_trace(False))  # 두 번째가 두 배
print("매번 reset:", collect_grad_trace(True))   # 두 번 모두 같음


# 원본 셀 121: reset은 기울기만 0 · parameter는 그대로
# 왜 · zero_()는 기울기만 지우고 값은 그대로 둔다는 것을 확인하기 위해
w.grad.zero_()  # 6-2의 w
b.grad.zero_()  # 6-2의 b
# 0 · 0 | 0 · 0
print("reset 뒤:", w.grad.item(), b.grad.item(), "| parameter는 여전히:", w.item(), b.item())

현재 loss: 17.0
w를 +0.01: 16.8703 | -0.01: 17.1302
b를 +0.01: 16.9201 | -0.01: 17.0801
w 쪽 변화율: -12.97
b 쪽 변화율: -7.99
reset 누락: [(1, -13.0, -8.0), (2, -26.0, -16.0)]
매번 reset: [(1, -13.0, -8.0), (2, -13.0, -8.0)]
reset 뒤: 0.0 0.0 | parameter는 여전히: 0.0 0.0


**실행 흐름**

1. 7-1 한 학습 단계
2. 7-2 no_grad · detach · item

In [44]:
# ---------- 7-1 ----------
# 원본 셀 127: 한 학습 단계 (학습률 0.1)
# 왜 · 한 학습 단계를 손계산과 같은 순서로 실행해 결과를 맞춰 보기 위해
w = torch.tensor(0.0, requires_grad=True)  # 새로 시작
b = torch.tensor(0.0, requires_grad=True)  # 새로 시작
step_size = 0.1                            # 학습률

# 왜 · 갱신 전 손실을 기록해 한 단계 뒤 줄었는지 비교하기 위해
prediction_before = w * x_train + b                                                  # 1. 예측
loss_before = ((prediction_before - target_train) ** 2).mean()                       # 2. 손실
loss_before.backward()                                                               # 3. 기울기
print(f"before: w={w.item():.3f}, b={b.item():.3f}, loss={loss_before.item():.3f}")  # 0 · 0 · 17
print(f"gradient: w.grad={w.grad.item():.3f}, b.grad={b.grad.item():.3f}")           # -13 · -8

with torch.no_grad():        # 4. 업데이트는 기록하지 않음
    w -= step_size * w.grad  # 0 - 0.1 × (-13)
    b -= step_size * b.grad  # 0 - 0.1 × (-8)
w.grad.zero_()               # 5. 다음 단계를 위해 비움
b.grad.zero_()               # 5. 다음 단계를 위해 비움

# 왜 · 갱신 후 새 예측으로 손실을 다시 계산하기 위해
prediction_after = w * x_train + b                                 # 6. 바뀐 값으로 새 예측
loss_after = ((prediction_after - target_train) ** 2).mean()       # 새 손실
print(f"after: w={w.item():.3f}, b={b.item():.3f}")                # 1.3 · 0.8
new_prediction = [round(v, 3) for v in prediction_after.tolist()]  # 보기 좋게 반올림
print("new prediction:", new_prediction, f"| new loss: {loss_after.item():.3f}")  # 17 → 1.685


# 원본 셀 125 손계산 기록표 답
# 왜 · 코드 결과와 손계산 결과가 같은지 검증하기 위해
manual_scaffold = {"new_w": 1.3, "new_b": 0.8,  # 0 - 0.1 × (-13) · 0 - 0.1 × (-8)
                   "prediction_1": 2.1, "prediction_2": 3.4, "new_loss": 1.685}
# 실행 값과 대조
print("손계산과 같음:", abs(manual_scaffold["new_loss"] - loss_after.item()) < 1e-6)


# ---------- 7-2 ----------
# 원본 셀 132: no_grad · detach · item의 책임
# 왜 · no_grad · detach · item이 각각 무엇을 끊는지 구분하기 위해
demo_value = torch.tensor(3.0, requires_grad=True)  # 기록되는 값
tracked_result = demo_value * 2                     # 기록됨
detached_result = tracked_result.detach()           # 텐서 하나를 기록에서 떼어 냄
python_number = tracked_result.item()               # Python 숫자
with torch.no_grad():                               # 블록 안 계산 전부 기록 안 함
    untracked_result = demo_value * 2               # 기록 없음

print("tracked:", type(tracked_result).__name__, tracked_result.requires_grad)     # Tensor True
print("detached:", type(detached_result).__name__, detached_result.requires_grad)  # Tensor False
print("item:", type(python_number).__name__)                                       # float
# Tensor False
print("no_grad 안 결과:", type(untracked_result).__name__, untracked_result.requires_grad)


# item() 결과에는 backward를 부를 수 없음 (원본 셀 131 질문)
# 왜 · 숫자로 꺼낸 값은 그래프와 끊긴다는 것을 확인하기 위해
print("item에 backward가 있나?", hasattr(python_number, "backward"))  # False

before: w=0.000, b=0.000, loss=17.000
gradient: w.grad=-13.000, b.grad=-8.000
after: w=1.300, b=0.800
new prediction: [2.1, 3.4] | new loss: 1.685
손계산과 같음: True
tracked: Tensor True
detached: Tensor False
item: float
no_grad 안 결과: Tensor False
item에 backward가 있나? False


**실행 흐름**

1. 7-3 반복 학습
2. 7-4 학습률 비교

In [45]:
# ---------- 7-3 ----------
# 원본 셀 135: 같은 순서를 4번 반복
# 왜 · 한 단계를 반복하면 손실이 계속 줄어드는지 확인하기 위해
w_loop = torch.tensor(0.0, requires_grad=True)                             # 시작 w
b_loop = torch.tensor(0.0, requires_grad=True)                             # 시작 b
for step in range(1, 5):                                                   # 4단계
    loss_loop = ((w_loop * x_train + b_loop - target_train) ** 2).mean()   # 예측 → 손실
    loss_loop.backward()                                                   # 기울기
    with torch.no_grad():                                                  # 업데이트
        w_loop -= 0.1 * w_loop.grad                                        # w 이동
        b_loop -= 0.1 * b_loop.grad                                        # b 이동
    w_loop.grad.zero_()                                                    # 비우기
    b_loop.grad.zero_()                                                    # 비우기
    fresh_loss = ((w_loop * x_train + b_loop - target_train) ** 2).mean()  # 바뀐 값의 성적
    print(f"step {step}: w={w_loop.item():.4f}, b={b_loop.item():.4f},"    # 바뀐 값
          f" fresh loss={fresh_loss.item():.6f}")


# 원본 셀 140: 세 책임이 빠진 loop → 같은 loss 재사용 에러
# 왜 · 학습 루프의 세 책임 중 하나라도 빠지면 생기는 에러를 재현하기 위해
broken_w = torch.tensor(0.0, requires_grad=True)                 # 새 w
broken_loss = ((broken_w * x_train - target_train) ** 2).mean()  # forward는 한 번만
broken_loss.backward()                                           # 1단계는 성공
try:
    broken_loss.backward()                                       # 2단계: 같은 loss를 다시
except RuntimeError as error:                                    # 에러를 잡아 멈추지 않게
    print("broken step 2:", type(error).__name__, "| w 그대로:", broken_w.item())  # 값도 안 바뀜


# 원본 셀 144 답: 한 단계의 실제 순서
# 왜 · 학습 한 단계의 실제 순서를 코드와 맞춰 정리하기 위해
student_order = ["prediction과 loss", "backward", "parameter update",  # 아직 없는 기울기는 못 씀
                 "gradient reset", "fresh prediction과 new loss"]
print(" → ".join(student_order))                                       # 한 줄로


# ---------- 7-4 ----------
# 원본 셀 146: 학습률만 바꾸는 실험 함수 (같은 데이터 · 같은 시작값)
# 왜 · 학습률만 바꾸고 나머지는 같게 해 공정하게 비교하기 위해
def run_step_size_trial(step_size, steps=8):                    # 학습률만 받는 실험
    trial_w = torch.tensor(0.0, requires_grad=True)             # 실험마다 새로
    trial_b = torch.tensor(0.0, requires_grad=True)             # 실험마다 새로
    history = []                                                # 업데이트 전 loss
    for _ in range(steps):                                      # steps번
        trial_loss = ((trial_w * x_train + trial_b - target_train) ** 2).mean()  # 손실
        history.append(trial_loss.item())                       # 기록
        trial_loss.backward()                                   # 기울기
        with torch.no_grad():                                   # 업데이트
            trial_w -= step_size * trial_w.grad                 # 이동 = 학습률 × 기울기
            trial_b -= step_size * trial_b.grad                 # b도 같은 규칙
        trial_w.grad.zero_()                                    # 비우기
        trial_b.grad.zero_()                                    # 비우기
    final_loss = ((trial_w * x_train + trial_b - target_train) ** 2).mean().item()  # 마지막 성적
    return history, trial_w.item(), trial_b.item(), final_loss  # 경로 · 도착점


# 원본 셀 148: 실험 A · B · C (8단계)
# 왜 · 학습률이 너무 작거나 크면 어떻게 되는지 세 실험으로 비교하기 위해
for label, size in {"실험 A": 0.001, "실험 B": 0.1, "실험 C": 0.5}.items():  # 학습률 3개
    history, final_w, final_b, final_loss = run_step_size_trial(size)        # 같은 조건
    print(f"{label} · 크기={size} | 처음 {history[0]:.3f}"                   # 경로 요약
          f" → 셋째 {history[2]:.3f} → 끝 {final_loss:.6g}")

step 1: w=1.3000, b=0.8000, fresh loss=1.685000
step 2: w=1.7100, b=1.0500, fresh loss=0.169250
step 3: w=1.8400, b=1.1270, fresh loss=0.019169
step 4: w=1.8819, b=1.1496, fresh loss=0.004246
broken step 2: RuntimeError | w 그대로: 0.0
prediction과 loss → backward → parameter update → gradient reset → fresh prediction과 new loss
실험 A · 크기=0.001 | 처음 17.000 → 셋째 16.539 → 끝 15.2288
실험 B · 크기=0.1 | 처음 17.000 → 셋째 0.169 → 끝 0.0023254
실험 C · 크기=0.5 | 처음 17.000 → 셋째 589.781 → 끝 2.46397e+07


**실행 흐름**

1. 중간 학습률 0.2 · 0.3으로 12단계 비교

In [46]:
# 원본 셀 150: 중간 학습률 0.2 · 0.3 (12단계)
# 왜 · 중간 학습률에서의 결과를 더 촘촘히 확인하기 위해
for size in [0.2, 0.3]:                                       # 두 후보
    history = run_step_size_trial(size, steps=12)[0]          # 경로만
    down = all(n <= p for p, n in zip(history, history[1:]))  # 매번 줄었나
    up = all(n >= p for p, n in zip(history, history[1:]))    # 매번 커졌나
    # 세 점
    print(f"lr={size}: 처음 {history[0]:.1f} · 둘째 {history[1]:.4g} · 마지막 {history[-1]:.4g}"
          f" | 단조 감소={down} · 단조 증가={up}")

lr=0.2: 처음 17.0 · 둘째 2.34 · 마지막 0.001533 | 단조 감소=True · 단조 증가=False


lr=0.3: 처음 17.0 · 둘째 18.97 · 마지막 56.63 | 단조 감소=False · 단조 증가=True


**실행 흐름**

1. 새 데이터로 손계산과 맞추기
2. 학습률 두 가지로 3단계씩
3. 이름이 바뀐 코드에서 역할 찾기

In [47]:
# ---------- 8. 종합 실습 ----------
# 원본 셀 137 답: 새 데이터 x = [1, 3] · 정답 [2, 6] · 시작 w = b = 0.5
# 왜 · 새 데이터에도 같은 학습 순서가 그대로 쓰이는지 손계산과 맞춰 보기 위해
transfer_x = torch.tensor([1.0, 3.0])                           # 입력
transfer_target = torch.tensor([2.0, 6.0])                      # 정답 (규칙 2x)
first_prediction_attempt, first_loss_attempt = [1.0, 2.0], 8.5  # 손계산 (예측 두 개 · 첫 손실)

# 왜 · 코드의 첫 예측 · 손실이 손계산과 같은지 확인하기 위해
actual_first_prediction = 0.5 * transfer_x + 0.5                               # 실제 첫 예측
actual_first_loss = ((actual_first_prediction - transfer_target) ** 2).mean()  # 실제 첫 손실
# 같음
print("첫 prediction 손계산 / 실제:", first_prediction_attempt, actual_first_prediction.tolist())
print("첫 loss 손계산 / 실제:", first_loss_attempt, actual_first_loss.item())  # 같음


# 학습률 0.03 · 0.05 두 가지로 3단계씩 (원본은 하나만 고르게 함)
# 왜 · 원본이 하나만 고르게 한 학습률을 둘 다 시험해 선택 근거를 보이기 위해
def transfer_run(step_size, steps=3):                                      # 학습률만 바꿔 3단계
    t_w = torch.tensor(0.5, requires_grad=True)                            # 시작 w
    t_b = torch.tensor(0.5, requires_grad=True)                            # 시작 b
    path = []                                                              # 단계별 fresh loss
    for _ in range(steps):                                                 # 3단계
        t_loss = ((t_w * transfer_x + t_b - transfer_target) ** 2).mean()  # 손실
        t_loss.backward()                                                  # 기울기
        with torch.no_grad():                                              # 업데이트
            t_w -= step_size * t_w.grad                                    # w 이동
            t_b -= step_size * t_b.grad                                    # b 이동
        t_w.grad.zero_()                                                   # 비우기
        t_b.grad.zero_()                                                   # 비우기
        # fresh loss
        path.append(round(((t_w * transfer_x + t_b - transfer_target) ** 2).mean().item(), 4))
    return round(t_w.item(), 4), round(t_b.item(), 4), path                # 도착점 · 경로


for step_size_choice in [0.03, 0.05]:  # 원본이 제시한 두 후보
    print(f"lr {step_size_choice}: w, b, fresh loss 경로 =", transfer_run(step_size_choice))  # 비교


# 원본 셀 155 · 156 답: 이름이 바뀐 코드에서 역할 찾기
# 왜 · 이름이 바뀐 코드에서도 같은 역할을 찾아 읽는 힘을 기르기 위해
features_new = torch.tensor([1.0, 2.0])                                    # 입력
labels_new = torch.tensor([3.0, 5.0])                                      # 정답
slope = torch.tensor(0.0, requires_grad=True)                              # parameter w
offset = torch.tensor(0.0, requires_grad=True)                             # parameter b
estimate = slope * features_new + offset                                   # prediction
score = ((estimate - labels_new) ** 2).mean()                              # loss
score.backward()                                                           # backward
with torch.no_grad():                                                      # update
    slope -= 0.1 * slope.grad                                              # w 이동
    offset -= 0.1 * offset.grad                                            # b 이동
slope.grad.zero_()                                                         # reset
offset.grad.zero_()                                                        # reset
check_estimate = slope * features_new + offset                             # fresh evidence
check_score = ((check_estimate - labels_new) ** 2).mean()                  # 새 손실
print(f"score {score.item():.3f} → check_score {check_score.item():.3f}")  # 7-1과 같은 숫자
# 역할 → 줄
code_roles = {"prediction": "estimate", "loss": "score", "backward": "score.backward()",
              "update": "-=", "reset": "zero_()", "fresh_evidence": "check_score"}
print(f"역할 연결: {sum(bool(v) for v in code_roles.values())}/{len(code_roles)}개 작성함")  # 6/6

첫 prediction 손계산 / 실제: [1.0, 2.0] [1.0, 2.0]
첫 loss 손계산 / 실제: 8.5 8.5
lr 0.03: w, b, fresh loss 경로 = (1.3122, 0.8021, [3.697, 1.6638, 0.8019])
lr 0.05: w, b, fresh loss 경로 = (1.5435, 0.8755, [1.625, 0.4237, 0.2098])
score 17.000 → check_score 1.685
역할 연결: 6/6개 작성함
